###### Imports and Settings

In [1]:
import pandas as pd
#import geopandas as gpd
import numpy as np
import requests
import io
import pickle
import matplotlib.pyplot as plt
from collections import deque
from functools import reduce
%matplotlib inline
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)

In [2]:
def percent(x, y):
    return x/y

In [3]:
data = pd.read_csv('../data/ACS20205YR.csv')

# Page by Page of CC Databook to start:

## Population + Projections Summary:

In [4]:
data['Population'] = data['agebysex_total_series']
data = data.drop(columns = 'agebysex_total_series')

## Health Crime Summary:  

### Health Insurance
+ Total Series  
+ Total No Health Insurance Coverage  
+ % Total with No Health Insurance Coverage  
+ Total with Health Insurance Coverage 
+ % Total with Health Insurance Coverage
+ Total with Health Insurance Coverage: Public Health Coverage  
+ % Covered Population: Public Health Coverage  
+ Total with Health Insurance Coverage: Private Health Coverage  
+ % Covered Population: Private Health Coverage  

In [5]:
#total series
data['HealthCoverage:Total Series'] = data['healthcoverage_total_healthcare_series']
#total no health insurance
nohealthcare = [data['healthcoverage_mu6_wo'],data['healthcoverage_m6to18_wo'],data['healthcoverage_m19to25_wo'],data['healthcoverage_m26to34_wo'],
                data['healthcoverage_m35to44_wo'],data['healthcoverage_m45to54_wo'],data['healthcoverage_m55to64_wo'],data['healthcoverage_m65to74_wo'],
                data['healthcoverage_m75+_wo'],data['healthcoverage_fu6_wo'],data['healthcoverage_f6to18_wo'],data['healthcoverage_f19to25_wo'],
                data['healthcoverage_f26to34_wo'],data['healthcoverage_f35to44_wo'],data['healthcoverage_f45to54_wo'],data['healthcoverage_f55to64_wo'],
                data['healthcoverage_f65to74_wo'],data['healthcoverage_f75+_wo']]
data['HealthCoverage:None'] = sum(nohealthcare)
#% of total with No Health Insurance Coverage
data['HealthCoverage%: None'] = percent(data['HealthCoverage:None'],data['HealthCoverage:Total Series'])
#total with health insurance
healthcare = [data['healthcoverage_mu6_w'],data['healthcoverage_m6to18_w'],data['healthcoverage_m19to25_w'],data['healthcoverage_m26to34_w'],
                data['healthcoverage_m35to44_w'],data['healthcoverage_m45to54_w'],data['healthcoverage_m55to64_w'],data['healthcoverage_m65to74_w'],
                data['healthcoverage_m75+_w'],data['healthcoverage_fu6_w'],data['healthcoverage_f6to18_w'],data['healthcoverage_f19to25_w'],
                data['healthcoverage_f26to34_w'],data['healthcoverage_f35to44_w'],data['healthcoverage_f45to54_w'],data['healthcoverage_f55to64_w'],
                data['healthcoverage_f65to74_w'],data['healthcoverage_f75+_w']]
data['HealthCoverage: With Healthcare Coverage'] = sum(healthcare)
#% of total with Health Insurance Coverage
data['HealthCoverage%: With Healthcare Coverage'] = data['HealthCoverage: With Healthcare Coverage']/data['HealthCoverage:Total Series']
# Total with Health Insurance: Public Coverage
public = [data['healthcoverage_mu6_wpublic'],data['healthcoverage_m6to18_wpublic'],data['healthcoverage_m19to25_wpublic'],data['healthcoverage_m26to34_wpublic'],
          data['healthcoverage_m35to44_wpublic'],data['healthcoverage_m45to54_wpublic'],data['healthcoverage_m55to64_wpublic'],
          data['healthcoverage_m65to74_wpublic'],data['healthcoverage_m75+_wpublic'],data['healthcoverage_fu6_wpublic'],
          data['healthcoverage_f6to18_wpublic'],data['healthcoverage_f19to25_wpublic'],data['healthcoverage_f26to34_wpublic'],
          data['healthcoverage_f35to44_wpublic'],data['healthcoverage_f45to54_wpublic'],data['healthcoverage_f55to64_wpublic'],
          data['healthcoverage_f65to74_wpublic'],data['healthcoverage_f75+_wpublic']]
data['HealthCoverage: With Public Healthcare Coverage'] = sum(public)
# % of covered with public
data['HealthCoverage%: With Public Healthcare Coverage'] = data['HealthCoverage: With Public Healthcare Coverage']/data['HealthCoverage: With Healthcare Coverage']
# Total with Health Insurance: Private Coverage
private = [data['healthcoverage_mu6_wprivate'],data['healthcoverage_m6to18_wprivate'],data['healthcoverage_m19to25_wprivate'],data['healthcoverage_m26to34_wprivate'],
          data['healthcoverage_m35to44_wprivate'],data['healthcoverage_m45to54_wprivate'],data['healthcoverage_m55to64_wprivate'],
          data['healthcoverage_m65to74_wprivate'],data['healthcoverage_m75+_wprivate'],data['healthcoverage_fu6_wprivate'],
          data['healthcoverage_f6to18_wprivate'],data['healthcoverage_f19to25_wprivate'],data['healthcoverage_f26to34_wprivate'],
          data['healthcoverage_f35to44_wprivate'],data['healthcoverage_f45to54_wprivate'],data['healthcoverage_f55to64_wprivate'],
          data['healthcoverage_f65to74_wprivate'],data['healthcoverage_f75+_wprivate']]
data['HealthCoverage: With Private Healthcare Coverage'] = sum(private)
# % of covered with private
data['HealthCoverage%: With Private Healthcare Coverage'] = data['HealthCoverage: With Private Healthcare Coverage']/data['HealthCoverage: With Healthcare Coverage']

cols = ['healthcoverage_total_healthcare_series','healthcoverage_total_mallhealthcare','healthcoverage_mu6_all','healthcoverage_mu6_w','healthcoverage_mu6_wo',
        'healthcoverage_m6to18_all','healthcoverage_m6to18_w','healthcoverage_m6to18_wo','healthcoverage_m19to25_all','healthcoverage_m19to25_w',
        'healthcoverage_m19to25_wo','healthcoverage_m26to34_all','healthcoverage_m26to34_w','healthcoverage_m26to34_wo','healthcoverage_m35to44_all',
        'healthcoverage_m35to44_w','healthcoverage_m35to44_wo','healthcoverage_m45to54_all','healthcoverage_m45to54_w','healthcoverage_m45to54_wo',
        'healthcoverage_m55to64_all','healthcoverage_m55to64_w','healthcoverage_m55to64_wo','healthcoverage_m65to74_all','healthcoverage_m65to74_w',
        'healthcoverage_m65to74_wo','healthcoverage_m75+_all','healthcoverage_m75+_w','healthcoverage_m75+_wo','healthcoverage_total_fallhealthcare',
        'healthcoverage_fu6_all','healthcoverage_fu6_w','healthcoverage_fu6_wo','healthcoverage_f6to18_all','healthcoverage_f6to18_w','healthcoverage_f6to18_wo',
        'healthcoverage_f19to25_all','healthcoverage_f19to25_w','healthcoverage_f19to25_wo','healthcoverage_f26to34_all','healthcoverage_f26to34_w',
        'healthcoverage_f26to34_wo','healthcoverage_f35to44_all','healthcoverage_f35to44_w','healthcoverage_f35to44_wo','healthcoverage_f45to54_all',
        'healthcoverage_f45to54_w','healthcoverage_f45to54_wo','healthcoverage_f55to64_all','healthcoverage_f55to64_w','healthcoverage_f55to64_wo',
        'healthcoverage_f65to74_all','healthcoverage_f65to74_w','healthcoverage_f65to74_wo','healthcoverage_f75+_all','healthcoverage_f75+_w',
        'healthcoverage_f75+_wo','healthcoverage_total_privatehealthcare_series','healthcoverage_total_mprivate','healthcoverage_mu6_privateall',
        'healthcoverage_mu6_wprivate','healthcoverage_mu6_woprivate','healthcoverage_m6to18_privateall','healthcoverage_m6to18_wprivate',
        'healthcoverage_m6to18_woprivate','healthcoverage_m19to25_privateall','healthcoverage_m19to25_wprivate','healthcoverage_m19to25_woprivate',
        'healthcoverage_m26to34_privateall','healthcoverage_m26to34_wprivate','healthcoverage_m26to34_woprivate','healthcoverage_m35to44_privateall',
        'healthcoverage_m35to44_wprivate','healthcoverage_m35to44_woprivate','healthcoverage_m45to54_privateall','healthcoverage_m45to54_wprivate',
        'healthcoverage_m45to54_woprivate','healthcoverage_m55to64_privateall','healthcoverage_m55to64_wprivate','healthcoverage_m55to64_woprivate',
        'healthcoverage_m65to74_privateall','healthcoverage_m65to74_wprivate','healthcoverage_m65to74_woprivate','healthcoverage_m75+_privateall',
        'healthcoverage_m75+_wprivate','healthcoverage_m75+_woprivate','healthcoverage_total_fprivate','healthcoverage_fu6_privateall','healthcoverage_fu6_wprivate',
        'healthcoverage_fu6_woprivate','healthcoverage_f6to18_privateall','healthcoverage_f6to18_wprivate','healthcoverage_f6to18_woprivate',
        'healthcoverage_f19to25_privateall','healthcoverage_f19to25_wprivate','healthcoverage_f19to25_woprivate','healthcoverage_f26to34_privateall',
        'healthcoverage_f26to34_wprivate','healthcoverage_f26to34_woprivate','healthcoverage_f35to44_privateall','healthcoverage_f35to44_wprivate',
        'healthcoverage_f35to44_woprivate','healthcoverage_f45to54_privateall','healthcoverage_f45to54_wprivate','healthcoverage_f45to54_woprivate',
        'healthcoverage_f55to64_privateall','healthcoverage_f55to64_wprivate','healthcoverage_f55to64_woprivate','healthcoverage_f65to74_privateall',
        'healthcoverage_f65to74_wprivate','healthcoverage_f65to74_woprivate','healthcoverage_f75+_privateall','healthcoverage_f75+_wprivate',
        'healthcoverage_f75+_woprivate','healthcoverage_total_publichealthcare_series','healthcoverage_total_mpublic','healthcoverage_mu6_publicall',
        'healthcoverage_mu6_wpublic','healthcoverage_mu6_wopublic','healthcoverage_m6to18_publicall','healthcoverage_m6to18_wpublic','healthcoverage_m6to18_wopublic',
        'healthcoverage_m19to25_publicall','healthcoverage_m19to25_wpublic','healthcoverage_m19to25_wopublic','healthcoverage_m26to34_publicall',
        'healthcoverage_m26to34_wpublic','healthcoverage_m26to34_wopublic','healthcoverage_m35to44_publicall','healthcoverage_m35to44_wpublic',
        'healthcoverage_m35to44_wopublic','healthcoverage_m45to54_publicall','healthcoverage_m45to54_wpublic','healthcoverage_m45to54_wopublic',
        'healthcoverage_m55to64_publicall','healthcoverage_m55to64_wpublic','healthcoverage_m55to64_wopublic','healthcoverage_m65to74_publicall',
        'healthcoverage_m65to74_wpublic','healthcoverage_m65to74_wopublic','healthcoverage_m75+_publicall','healthcoverage_m75+_wpublic',
        'healthcoverage_m75+_wopublic','healthcoverage_total_fpublic','healthcoverage_fu6_publicall','healthcoverage_fu6_wpublic','healthcoverage_fu6_wopublic',
        'healthcoverage_f6to18_publicall','healthcoverage_f6to18_wpublic','healthcoverage_f6to18_wopublic','healthcoverage_f19to25_publicall',
        'healthcoverage_f19to25_wpublic','healthcoverage_f19to25_wopublic','healthcoverage_f26to34_publicall','healthcoverage_f26to34_wpublic',
        'healthcoverage_f26to34_wopublic','healthcoverage_f35to44_publicall','healthcoverage_f35to44_wpublic','healthcoverage_f35to44_wopublic',
        'healthcoverage_f45to54_publicall','healthcoverage_f45to54_wpublic','healthcoverage_f45to54_wopublic','healthcoverage_f55to64_publicall',
        'healthcoverage_f55to64_wpublic','healthcoverage_f55to64_wopublic','healthcoverage_f65to74_publicall','healthcoverage_f65to74_wpublic',
        'healthcoverage_f65to74_wopublic','healthcoverage_f75+_publicall','healthcoverage_f75+_wpublic','healthcoverage_f75+_wopublic']
data = data.drop(columns = cols)

## Age Sex Race Summary:

### Age Brackets
+ M and F U5, 5 to 9, 10 to 14, 15 to 17, 18 to 24, 25 to 34, 35 to 44, 45 to 54, 55 to 64, 65 to 74, 75 to 84, 85+  
+ age brackets: under 18, 18 to 54, 55+  
+ 65+ 

In [6]:
#small groups m and f
data['Male Under 5'] = data['age_m_u5']
data['Female Under 5'] = data['age_f_u5']
data['Male 5 to 9'] = data['age_m_5to9']
data['Female 5 to 9'] = data['age_f_5to9']
data['Male 10 to 14'] = data['age_m_10to14']
data['Female 10 to 14'] = data['age_f_10to14']
data['Male 15 to 17'] = data['age_m_15to17']
data['Female 15 to 17'] = data['age_f_15to17']
data['Male 18 to 24'] = data['age_m_18to19']+data['age_m_20']+data['age_m_21']+data['age_m_22to24']
data['Female 18 to 24'] = data['age_f_18to19']+data['age_f_20']+data['age_f_21']+data['age_f_22to24']
data['Male 25 to 34'] = data['age_m_25to29']+data['age_m_30to34']
data['Female 25 to 34'] = data['age_f_25to29']+data['age_f_30to34']
data['Male 35 to 44'] = data['age_m_35to39']+data['age_m_40to44']
data['Female 35 to 44'] = data['age_f_35to39']+data['age_f_40to44']
data['Male 45 to 54'] = data['age_m_45to49']+data['age_m_50to54']
data['Female 45 to 54'] = data['age_f_45to49']+data['age_f_50to54']
data['Male 55 to 64'] = data['age_m_55to59']+data['age_m_60to61']+data['age_m_62to64']
data['Female 55 to 64'] = data['age_f_55to59']+data['age_f_60to61']+data['age_f_62to64']
data['Male 65 to 74'] = data['age_m_65to66']+data['age_m_67to69']+data['age_m_70to74']
data['Female 65 to 74'] = data['age_f_65to66']+data['age_f_67to69']+data['age_f_70to74']
data['Male 75 to 84'] = data['age_m_75to79']+data['age_m_80to84']
data['Female 75 to 84'] = data['age_f_75to79']+data['age_f_80to84']
data['Male 85 and Older'] = data['age_m_85+']
data['Female 85 and Older'] = data['age_f_85+']
#age brackets
u18list = [data['Male Under 5'],data['Female Under 5'],data['Male 5 to 9'],data['Female 5 to 9'],data['Male 10 to 14'],data['Female 10 to 14'],data['Male 15 to 17'],
           data['Female 15 to 17']]
data['Age:Under 18'] = sum(u18list)
eighteento54list = [data['Male 18 to 24'],data['Female 18 to 24'],data['Male 25 to 34'],data['Female 25 to 34'],data['Male 35 to 44'],data['Female 35 to 44'],
              data['Male 45 to 54'],data['Female 45 to 54']]
data['Age:18 to 24'] = sum(eighteento54list)
fifty5pluslist = [data['Male 55 to 64'],data['Female 55 to 64'],data['Male 65 to 74'],data['Female 65 to 74'],data['Male 75 to 84'],data['Female 75 to 84'],
                  data['Male 85 and Older'],data['Female 85 and Older']]
data['Age:55 and Older'] = sum(fifty5pluslist)
#65+
sixty5pluslist = [data['Male 65 to 74'],data['Female 65 to 74'],data['Male 75 to 84'],data['Female 75 to 84'],data['Male 85 and Older'],data['Female 85 and Older']]
data['Age:65 and Older'] = sum(sixty5pluslist)
cols = ['age_total_male','age_total_female','age_m_u5','age_m_5to9','age_m_10to14','age_m_15to17','age_m_18to19','age_m_20','age_m_21','age_m_22to24','age_m_25to29','age_m_30to34','age_m_35to39',
        'age_m_40to44','age_m_45to49','age_m_50to54','age_m_55to59','age_m_60to61','age_m_62to64','age_m_65to66','age_m_67to69','age_m_70to74','age_m_75to79',
        'age_m_80to84','age_m_85+','age_f_u5','age_f_5to9','age_f_10to14','age_f_15to17','age_f_18to19','age_f_20','age_f_21','age_f_22to24','age_f_25to29',
        'age_f_30to34','age_f_35to39','age_f_40to44','age_f_45to49','age_f_50to54','age_f_55to59','age_f_60to61','age_f_62to64','age_f_65to66','age_f_67to69',
        'age_f_70to74','age_f_75to79','age_f_80to84','age_f_85+']
data = data.drop(columns = cols)

### Race/Ethnicity #s   
+ White Alone  
+ Black/African American Alone  
+ American Indian Alaska Native Alone  
+ Asian Alone  
+ Native Hawaiian/Other Pacific Islander Alone  
+ Some Other Race Alone  
+ Two or More Races  
+ Hispanic/Latino  
+ White, Not Hispanic/Latino  
+ Total Minority (non-white, non Hispanic/Latino)

In [7]:
#White Alone
data['White Alone'] = data['raceeth_white_alone']
data['White Alone %'] = data['White Alone']/data['Population']
#Black or African American Alone 
data['Black or African American Alone'] = data['raceeth_blackafricanamerican_alone']
data['Black or African American Alone %'] = data['Black or African American Alone']/data['Population']
#American Indian and Alaska Native Alone
data['American Indian Alaska Native Alone'] = data['raceeth_americanindianalaskanative_alone']
data['American Indian Alaska Native Alone %'] = data['American Indian Alaska Native Alone']/data['Population']
#Asian Alone
data['Asian Alone'] = data['raceeth_asian_alone']
data['Asian Alone %'] = data['Asian Alone']/data['Population']
#Native Hawaiian Other Pacific Islander Alone
data['Native Hawaiian Other Pacific Islander Alone'] = data['raceeth_nativehawaiianotherpacificislander_alone']
data['Native Hawaiian Other Pacific Islander Alone %'] = data['Native Hawaiian Other Pacific Islander Alone']/data['Population']
#Some Other Race Alone
data['Some Other Race Alone'] = data['raceeth_someotherrace_alone']
data['Some Other Race Alone %'] = data['Some Other Race Alone']/data['Population']
#Two or More Races
data['Two or More Races'] = data['raceeth_twoormoreraces_alone']
data['Two or More Races %'] = data['Two or More Races']/data['Population']
#Hispanic or Latino
data['Hispanic or Latino'] = data['raceeth_hispanicorlatino']
data['Hispanic or Latino %'] = data['Hispanic or Latino']/data['Population']
#Data Minority
data['Minority'] = data['Population'] - data['raceeth_whitealone_nothispanicorlatino']
data['Minority %'] = data['Minority']/data['Population']
cols = ['raceeth_white_alone','raceeth_blackafricanamerican_alone','raceeth_americanindianalaskanative_alone','raceeth_asian_alone',
        'raceeth_nativehawaiianotherpacificislander_alone','raceeth_someotherrace_alone','raceeth_twoormoreraces_alone','raceeth_hispanicorlatino',
        'raceeth_whitealone_nothispanicorlatino']
data = data.drop(columns = cols)

## Household Summary:  

### Households by Household Type  
+ Total Households  
+ Family Households  
+ Family Households: Married-Couple Family  
+ Family Households: Other Family  
+ Family Households: Other Family: Male Householder no Wife Present  
+ Family Households: Other Family: Female Householder no Husband Present  
+ Nonfamily Households  
+ Nonfamily Household: Male Householder  
+ Nonfamily Household: Female Householder  

*They don't have the nonfamily male or female - replaced by Householder alone or not alone*

In [8]:
data['Total Households'] = data['hhtype_total_series']
data['Family Households'] = data['hhtype_familyhh']
data['Family Households: Married Couple Family'] = data['hhtype_familyhh_marriedcouplefam']
data['Family Households: Not Married Couple Family'] = data['hhtype_familyhh_otherfam']
data['Family Households: Not Married Couple: Male no Spouse'] = data['hhtype_familyhh_malenospouse']
data['Family Households: Not Married Couple: Female no Spouse'] = data['hhtype_familyhh_femalenospouse']
data['Nonfamily Households'] = data['hhtype_nonfamhh']
data['Nonfamily Households: Householder Alone'] = data['hhtype_nonfamhh_householderalone']
data['Nonfamily Households: Householder not Alone'] = data['hhtype_nonfamhh_householdernotalone']

cols = ['hhtype_total_series','hhtype_familyhh','hhtype_familyhh_marriedcouplefam','hhtype_familyhh_otherfam','hhtype_familyhh_malenospouse',
        'hhtype_familyhh_femalenospouse','hhtype_nonfamhh','hhtype_nonfamhh_householderalone','hhtype_nonfamhh_householdernotalone']
data = data.drop(columns = cols)

### Average Household Size 
Median - drop for incorporated and unincorporated

In [9]:
data['Average Household Size'] = data['hhsize_avg']
data = data.drop(columns = ['hhsize_avg'])

###  Median Household Income  
Median - drop for incorporated and unincorporated

In [10]:
data['Median Household Income'] = data['hhincome_median']
data = data.drop(columns = ['hhincome_median'])

### Households by Household Income  
+ Less than 10,000  
+ 10,000 to 14,999  
+ 15,000 to 19,999  
+ 20,000 to 24,999 
+ 25,000 to 29,999  
+ 30,000 to 34,999  
+ 35,000 to 39,999  
+ 40,000 to 44,999  
+ 50,000 to 59,999  
+ 60,000 to 74,999  
+ 75,000 to 99,999  
+ 100,000 to 124,999  
+ 125,000 to 149,999  
+ 150,000 to 199,999  
+ 200,000 or More

In [11]:
data['HHIncome:Total Households'] = data['hhincome_total_series']
data['HHIncome:Less than 10,000'] = data['hhincome_lessthan10000']
data['HHIncome%:Less than 10,000'] = percent(data['HHIncome:Less than 10,000'], data['HHIncome:Total Households'])
data['HHIncome:10 to 14,999'] = data['hhincome_10to14999']
data['HHIncome%:10 to 14,999'] = percent(data['HHIncome:10 to 14,999'], data['HHIncome:Total Households'])
data['HHIncome:15 to 19,999'] = data['hhincome_15to19999']
data['HHIncome%:15 to 19,999'] = percent(data['HHIncome:15 to 19,999'], data['HHIncome:Total Households'])
data['HHIncome:20 to 24,999'] = data['hhincome_20to24999']
data['HHIncome%:20 to 24,999'] = percent(data['HHIncome:20 to 24,999'], data['HHIncome:Total Households'])
data['HHIncome:25 to 29,999'] = data['hhincome_25to29999']
data['HHIncome%:25 to 29,999'] = percent(data['HHIncome:25 to 29,999'], data['HHIncome:Total Households'])
data['HHIncome:30 to 34,999'] = data['hhincome_30to34999']
data['HHIncome:%30 to 34,999'] = percent(data['HHIncome:30 to 34,999'], data['HHIncome:Total Households'])
data['HHIncome:35 to 39,999'] = data['hhincome_35to39999']
data['HHIncome%:35 to 39,999'] = percent(data['HHIncome:35 to 39,999'], data['HHIncome:Total Households'])
data['HHIncome:40 to 44,999'] = data['hhincome_40to44999']
data['HHIncome%:40 to 44,999'] = percent(data['HHIncome:40 to 44,999'], data['HHIncome:Total Households'])
data['HHIncome:45 to 49,999'] = data['hhincome_45to49999']
data['HHIncome%:45 to 49,999'] = percent(data['HHIncome:45 to 49,999'], data['HHIncome:Total Households'])
data['HHIncome:50 to 59,999'] = data['hhincome_50to59999']
data['HHIncome%:50 to 59,999'] = percent(data['HHIncome:50 to 59,999'], data['HHIncome:Total Households'])
data['HHIncome:60 to 74,999'] = data['hhincome_60to74999']
data['HHIncome%:60 to 74,999'] = percent(data['HHIncome:60 to 74,999'], data['HHIncome:Total Households'])
data['HHIncome:75 to 99,999'] = data['hhincome_75to99999']
data['HHIncome%:75 to 99,999'] = percent(data['HHIncome:75 to 99,999'], data['HHIncome:Total Households'])
data['HHIncome:100 to 124,999'] = data['hhincome_100to124999']
data['HHIncome%:100 to 124,999'] = percent(data['HHIncome:100 to 124,999'], data['HHIncome:Total Households'])
data['HHIncome:125 to 149,999'] = data['hhincome125to149999']
data['HHIncome%:125 to 149,999'] = percent(data['HHIncome:125 to 149,999'], data['HHIncome:Total Households'])
data['HHIncome:150 to 199,999'] = data['hhincome150to199999']
data['HHIncome%:150 to 199,999'] = percent(data['HHIncome:150 to 199,999'], data['HHIncome:Total Households'])
data['HHIncome:200K or More'] = data['hhincome200ormore']
data['HHIncome%:200K or More'] = percent(data['HHIncome:200K or More'], data['HHIncome:Total Households'])

cols = ['hhincome_total_series','hhincome_lessthan10000','hhincome_10to14999','hhincome_15to19999','hhincome_20to24999','hhincome_25to29999',
        'hhincome_30to34999','hhincome_35to39999','hhincome_40to44999','hhincome_45to49999','hhincome_50to59999','hhincome_60to74999','hhincome_75to99999',
        'hhincome_100to124999','hhincome125to149999','hhincome150to199999','hhincome200ormore']
data = data.drop(columns = cols)

## Education and Work Summary:  

### Educational Attainment Population 25+, and Percentages
+ Population 25+  
+ Less than High School 25+  
+ High School Graduate or Equivalency 25+  
+ Some College 25+  
+ Associates  
+ Bachelor's Degree 25+  
+ Master's Degree 25+  
+ Professional School Degree 25+  
+ Doctorate Degree 25+  

In [12]:
data['Ed:Population 25+ Educational Attainment'] = data['attainment_total_over25_series']
lesshighschoollist = [data['attainment_noschooling'],data['attainment_nurseryschool'],data['attainment_kindergarten'],data['attainment_1stgrade'],
                      data['attainment_2ndgrade'],data['attainment_3rdgrade'],data['attainment_4thgrade'],data['attainment_5thgrade'],data['attainment_6thgrade'],
                      data['attainment_7thgrade'],data['attainment_8thgrade'],data['attainment_9thgrade'],data['attainment_10thgrade'],data['attainment_11thgrade'],
                      data['attainment_12thgradenodiploma']]
data['Ed:Less than High School'] = sum(lesshighschoollist)
data['Ed%:Less than High School'] = percent(data['Ed:Less than High School'], data['Ed:Population 25+ Educational Attainment'])
data['Ed:High School Graduate or Equivalency'] = data['attainment_regularhighschooldiploma']+data['attainment_gedoralternativecredential']
data['Ed%:High School Graduate or Equivalency'] = percent(data['Ed:High School Graduate or Equivalency'], data['Ed:Population 25+ Educational Attainment'])
highormore = [data['attainment_regularhighschooldiploma'],data['attainment_gedoralternativecredential'],data['attainment_somecollegelessthan1year'],
              data['attainment_somecollege1ormoreyearsnodegree'],data['attainment_associatesdegree'],data['attainment_bachelorsdegree'],
              data['attainment_mastersdegree'],data['attainment_professionalschooldegree'],data['attainment_doctoratedegree']]
data['Ed%:High School Graduate or More'] = percent(sum(highormore),data['Ed:Population 25+ Educational Attainment'])
data['Ed:Some College'] = data['attainment_somecollegelessthan1year']+data['attainment_somecollege1ormoreyearsnodegree']
data['Ed%:Some College'] = percent(data['Ed:Some College'], data['Ed:Population 25+ Educational Attainment'])
somecollegeormore = [data['attainment_somecollege1ormoreyearsnodegree'],data['attainment_associatesdegree'],data['attainment_bachelorsdegree'],
                     data['attainment_mastersdegree'],data['attainment_professionalschooldegree'],data['attainment_doctoratedegree']]
data['Ed%:Some College or More'] = percent(sum(somecollegeormore),data['Ed:Population 25+ Educational Attainment'])
data['Ed:Associates'] = data['attainment_associatesdegree']
data['Ed%:Associates'] = percent(data['Ed:Associates'], data['Ed:Population 25+ Educational Attainment'])
data['Ed:Bachelors'] = data['attainment_bachelorsdegree']
data['Ed%:Bachelors'] = percent(data['Ed:Bachelors'], data['Ed:Population 25+ Educational Attainment'])
bachormore = [data['attainment_bachelorsdegree'],data['attainment_mastersdegree'],data['attainment_professionalschooldegree'],data['attainment_doctoratedegree']]
data['Ed%:Bachelors or More'] = percent(sum(bachormore),data['Ed:Population 25+ Educational Attainment'])
data['Ed:Masters'] = data['attainment_mastersdegree']
mastersormore = [data['attainment_mastersdegree'],data['attainment_professionalschooldegree'],data['attainment_doctoratedegree']]
data['Ed%:Masters or More'] = percent(sum(mastersormore),data['Ed:Population 25+ Educational Attainment'])
data['Ed%:Masters'] = percent(data['Ed:Masters'], data['Ed:Population 25+ Educational Attainment'])
data['Ed:Professional School Degree'] = data['attainment_professionalschooldegree']
data['Ed%:Professional School Degree'] = percent(data['Ed:Professional School Degree'], data['Ed:Population 25+ Educational Attainment'])
proformore = [data['attainment_mastersdegree'],data['attainment_professionalschooldegree'],data['attainment_doctoratedegree']]
data['Ed%:Professional School Degree or More'] = percent(sum(proformore),data['Ed:Population 25+ Educational Attainment'])
data['Ed:Doctorate Degree'] = data['attainment_doctoratedegree']
data['Ed%:Doctorate Degree'] = percent(data['Ed:Doctorate Degree'], data['Ed:Population 25+ Educational Attainment'])

cols = ['attainment_total_over25_series','attainment_noschooling','attainment_nurseryschool','attainment_kindergarten','attainment_1stgrade','attainment_2ndgrade',
        'attainment_3rdgrade','attainment_4thgrade','attainment_5thgrade','attainment_6thgrade','attainment_7thgrade','attainment_8thgrade','attainment_9thgrade',
        'attainment_10thgrade','attainment_11thgrade','attainment_12thgradenodiploma','attainment_regularhighschooldiploma','attainment_gedoralternativecredential',
        'attainment_somecollegelessthan1year','attainment_somecollege1ormoreyearsnodegree','attainment_associatesdegree','attainment_bachelorsdegree',
        'attainment_mastersdegree','attainment_professionalschooldegree','attainment_doctoratedegree']
data = data.drop(columns = cols)

### Industry Employment 16+, and Percentages  
+ Employed Civilians 16+  
+ Agriculture, forestry, fishing and hunting, and mining  
+ Construction  
+ Manufacturing  
+ Wholesale Trade  
+ Retail Trade  
+ Transportation and warehousing, and utilities  
+ Information  
+ Finance, insurance, real estate and rental and leasing  
+ Professional, scientific, management, administrative, and waste management services  
+ Educational, health and social services  
+ Arts, entertainment, recreation, accommodation and food services  
+ Other services (except public administration)  
+ Public administration

In [13]:
data['Ind:Employed Civilians 16+'] = data['industry_sexbyindustrycivilianpop_total_series']
aglist = [data['industry_m_agriculture_forestry_fishing_hunting_mining'],data['industry_f_agriculture_forestry_fishing_hunting_mining']]
data['Ind:Agriculture, Forestry, Fishing and Hunting, and Mining'] = sum(aglist)
data['Ind%:Agriculture, Forestry, Fishing and Hunting, and Mining']=percent(data['Ind:Agriculture, Forestry, Fishing and Hunting, and Mining'],
                                                                            data['Ind:Employed Civilians 16+'])
data['Ind:Construction'] = data['industry_m_construction']+data['industry_f_construction']
data['Ind%:Construction']=percent(data['Ind:Construction'],data['Ind:Employed Civilians 16+'])
data['Ind:Manufacturing'] = data['industry_m_manufacturing'] + data['industry_f_manufacturing']
data['Ind%:Manufacturing']=percent(data['Ind:Manufacturing'],data['Ind:Employed Civilians 16+'])
data['Ind:Wholesale Trade'] = data['industry_m_wholesaletrade']+data['industry_f_wholesaletrade']
data['Ind%:Wholesale Trade']=percent(data['Ind:Wholesale Trade'],data['Ind:Employed Civilians 16+'])
data['Ind:Retail Trade'] = data['industry_m_retailtrade']+data['industry_f_retailtrade']
data['Ind%:Retail Trade']=percent(data['Ind:Retail Trade'],data['Ind:Employed Civilians 16+'])
transware = [data['industry_m_transportation_warehousing_utilities'],data['industry_f_transportation_warehousing_utilities']]
data['Ind:Transportation and Warehousing, Utilities'] = sum(transware)
data['Ind%:Transportation and Warehousing, Utilities']=percent(data['Ind:Transportation and Warehousing, Utilities'],data['Ind:Employed Civilians 16+'])
data['Ind:Information'] = data['industry_m_information']+data['industry_f_information']
data['Ind%:Information']=percent(data['Ind:Information'],data['Ind:Employed Civilians 16+'])
finins = [data['industry_m_finance_insurance_realestate_rental'],data['industry_f_finance_insurance_realestate_rental']]
data['Ind:Finance, Insurance, Real Estate and Rental and Leasing'] = sum(finins)
data['Ind%:Finance, Insurance, Real Estate and Rental and Leasing']=percent(data['Ind:Finance, Insurance, Real Estate and Rental and Leasing'],
                                                                            data['Ind:Employed Civilians 16+'])
profsci = [data['industry_m_professional_scientific_managementadmin_waste'],data['industry_f_professional_scientific_managementadmin_waste']]
data['Ind:Professional, Scientific, Management, Administrative, and Waste Management Services']= sum(profsci)
data['Ind%:Professional, Scientific, Management, Administrative, and Waste Management Services']=percent(data['Ind:Professional, Scientific, Management, Administrative, and Waste Management Services'],
                                                                                                         data['Ind:Employed Civilians 16+'])
edhealth = [data['industry_m_educational_healthcare_socialassistance'],data['industry_f_educational_healthcare_socialassistance']]
data['Ind:Educational, Health and Social Services'] = sum(edhealth)
data['Ind%:Educational, Health and Social Services']=percent(data['Ind:Educational, Health and Social Services'],data['Ind:Employed Civilians 16+'])
artsrec = [data['industry_m_arts_entertainmentrec_accommodation_food'],data['industry_f_arts_entertainmentrec_accommodation_food']]
data['Ind:Arts, Entertainment, Recreation, Accommodation and Food Services'] = sum(artsrec)
data['Ind%:Arts, Entertainment, Recreation, Accommodation and Food Services']=percent(data['Ind:Arts, Entertainment, Recreation, Accommodation and Food Services'],
                                                                                      data['Ind:Employed Civilians 16+'])
data['Ind:Other Services (Except Public Administration)']=data['industry_m_otherservices_notpublicadmin']+data['industry_f_otherservices_notpublicadmin']
data['Ind%:Other Services (Except Public Administration)']=percent(data['Ind:Other Services (Except Public Administration)'],data['Ind:Employed Civilians 16+'])
data['Ind:Public Administration'] = data['industry_m_publicadmin']+data['industry_f_publicadmin']
data['Ind%:Public Administration']=percent(data['Ind:Public Administration'],data['Ind:Employed Civilians 16+'])

cols = ['industry_sexbyindustrycivilianpop_total_series','industry_sexbyindustrycivilianpop_totalmale','industry_m_agriculture_forestry_fishing_hunting_mining',
        'industry_m_agmining_agriculture_forestry_fishing_hunting','industry_m_agmining_mining_quarrying_oilandgas','industry_m_construction',
        'industry_m_manufacturing','industry_m_wholesaletrade','industry_m_retailtrade','industry_m_transportation_warehousing_utilities',
        'industry_m_transut_transportation_warehousing','industry_m_transut_utilities','industry_m_information','industry_m_finance_insurance_realestate_rental',
        'industry_m_finrent_finance_insurance','industry_m_finrent_realestate_rental','industry_m_professional_scientific_managementadmin_waste',
        'industry_m_profwaste_professional_scientific_technicalservices','industry_m_profwaste_managementcompaniesenterprises','industry_m_adminandsupportandwaste',
        'industry_m_educational_healthcare_socialassistance','industry_m_edsocial_educational','industry_m_edsocial_healthcare_socialassistance',
        'industry_m_arts_entertainmentrec_accommodation_food','industry_m_artsaccom_arts_entertainmentrec','industry_m_artsaccom_accommodation_food',
        'industry_m_otherservices_notpublicadmin','industry_m_publicadmin','industry_sexbyindustrycivilianpop_totalfemale',
        'industry_f_agriculture_forestry_fishing_hunting_mining','industry_f_agmining_agriculture_forestry_fishing_hunting',
        'industry_f_agmining_mining_quarrying_oilandgas','industry_f_construction','industry_f_manufacturing','industry_f_wholesaletrade','industry_f_retailtrade',
        'industry_f_transportation_warehousing_utilities','industry_f_transut_transportation_warehousing','industry_f_transut_utilities','industry_f_information',
        'industry_f_finance_insurance_realestate_rental','industry_f_finrent_finance_insurance','industry_f_finrent_realestate_rental',
        'industry_f_professional_scientific_managementadmin_waste','industry_f_profwaste_professional_scientific_technicalservices',
        'industry_f_profwaste_managementcompaniesenterprises','industry_f_adminandsupportandwaste','industry_f_educational_healthcare_socialassistance',
        'industry_f_edsocial_educational','industry_f_edsocial_healthcare_socialassistance','industry_f_arts_entertainmentrec_accommodation_food',
        'industry_f_artsaccom_arts_entertainmentrec','industry_f_artsaccom_accommodation_food','industry_f_otherservices_notpublicadmin','industry_f_publicadmin']
data = data.drop(columns = cols)

### Occupational Employment 16+, and Percentages  
+ Employed Civilians 16+  
+ Management, Business, and Financial Operations 
+ Professional and Related Occupations  
+ Healthcare Support Operations  
+ Protective Services Occupations  
+ Food Preparation and Serving Related Occupations  
+ Building and Grounds Cleaning and Maintenance Occupations  
+ Personal Care and Service Occupations  
+ Sales and Related Occupations  
+ Office and Administrative Support Occupations  
+ Farming, Fishing, and Forestry Occupations  
+ Construction, Extraction, and Maintenance Occupations  
+ Production Occupations  
+ Transportation and Material Moving Occupations

In [14]:
data['Employed Civilians 16+ Occupational'] = data['occupation_sexbyoccupationcivilianpop_total_series']
manbusfin = [data['occupation_m_management'],data['occupation_f_management'],data['occupation_m_businessandfinancial'],
             data['occupation_f_businessandfinancial']]
data['Occ:Management, Business, and Financial Operations'] = sum(manbusfin)
data['Occ%:Management, Business, and Financial Operations'] = percent(data['Occ:Management, Business, and Financial Operations'],
                                                                      data['Employed Civilians 16+ Occupational'])
profandrelated = [data['occupation_m_businessandfinancial'], data['occupation_m_computerandmathematical'], data['occupation_m_architectureandengineering'],
                  data['occupation_m_lifephysicalsocialscience'], data['occupation_m_communityandsocialservices'], data['occupation_m_legal'],
                  data['occupation_m_educationtrainingandlibrary'], data['occupation_m_artsdesignentertainmentsportsandmedia'], 
                  data['occupation_m_healthcarepractitionersandtechnical'], data['occupation_f_businessandfinancial'], 
                  data['occupation_f_computerandmathematical'], data['occupation_f_architectureandengineering'], data['occupation_f_architectureandengineering'],
                  data['occupation_f_lifephysicalsocialscience'], data['occupation_f_communityandsocialservices'], data['occupation_f_legal'],
                  data['occupation_f_educationtrainingandlibrary'], data['occupation_f_artsdesignentertainmentsportsandmedia'],
                  data['occupation_f_healthcarepractitionersandtechnical']]
data['Occ:Professional and Related'] = sum(profandrelated)
data['Occ%:Professional and Related'] = percent(data['Occ:Professional and Related'],data['Employed Civilians 16+ Occupational'])
data['Occ:Healthcare Support'] = data['occupation_m_healthcaresupport']+data['occupation_f_healthcaresupport']
data['Occ%:Healthcare Support'] = percent(data['Occ:Healthcare Support'],data['Employed Civilians 16+ Occupational'])
data['Occ:Protective Services'] = data['occupation_m_protectiveservice']+data['occupation_f_protectiveservice']
data['Occ%:Protective Services'] = percent(data['Occ:Protective Services'],data['Employed Civilians 16+ Occupational'])
data['Occ:Food Preparation and Serving Related'] = data['occupation_m_foodprepandserving']+data['occupation_f_foodprepandserving']
data['Occ%:Food Preparation and Serving Related'] = percent(data['Occ:Food Preparation and Serving Related'],data['Employed Civilians 16+ Occupational'])
buildinggrounds = [data['occupation_m_buildingandgroundscleaningandmaintenance'],data['occupation_f_buildingandgroundscleaningandmaintenance']]
data['Occ:Building and Grounds Cleaning and Maintenance'] = sum(buildinggrounds)
data['Occ%:Building and Grounds Cleaning and Maintenance'] = percent(data['Occ:Building and Grounds Cleaning and Maintenance'],
                                                                     data['Employed Civilians 16+ Occupational'])
data['Occ:Personal Care and Service'] = data['occupation_m_personalcareandservice'] + data['occupation_f_personalcareandservice']
data['Occ%:Personal Care and Service'] = percent(data['Occ:Personal Care and Service'],data['Employed Civilians 16+ Occupational'])
data['Occ:Sales and Related'] = data['occupation_m_salesandrelated']+data['occupation_f_salesandrelated']
data['Occ%:Sales and Related'] = percent(data['Occ:Sales and Related'],data['Employed Civilians 16+ Occupational'])
data['Occ:Office and Administrative Support'] = data['occupation_m_officeandadmin'] + data['occupation_f_officeandadmin']
data['Occ%:Office and Administrative Support'] = percent(data['Occ:Office and Administrative Support'],data['Employed Civilians 16+ Occupational'])
data['Occ:Farming, Fishing, and Forestry'] = data['occupation_m_farmingfishingandforestry']+data['occupation_f_farmingfishingandforestry']
data['Occ%:Farming, Fishing, and Forestry'] = percent(data['Occ:Farming, Fishing, and Forestry'],data['Employed Civilians 16+ Occupational'])
constexmaint = [data['occupation_m_constructionandextraction'],data['occupation_m_installationmaintenanceandrepair'],data['occupation_f_constructionandextraction'],
               data['occupation_f_installationmaintenanceandrepair']]
data['Occ:Construction, Extraction, and Maintenance'] = sum(constexmaint)
data['Occ%:Construction, Extraction, and Maintenance'] = percent(data['Occ:Construction, Extraction, and Maintenance'],data['Employed Civilians 16+ Occupational'])
data['Occ:Production'] = data['occupation_m_production']+data['occupation_f_production']
data['Occ%:Production'] = percent(data['Occ:Production'],data['Employed Civilians 16+ Occupational'])
transpomat = [data['occupation_m_transportation'],data['occupation_f_transportation'],data['occupation_m_materialmoving'],data['occupation_f_materialmoving']]
data['Occ:Transportation and Material Moving'] = sum(transpomat)
data['Occ%:Transportation and Material Moving'] = percent(data['Occ:Transportation and Material Moving'],data['Employed Civilians 16+ Occupational'])

cols = ['occupation_sexbyoccupationcivilianpop_total_series','occupation_sexbyoccupationcivilianpop_totalmale','occupation_m_management',
        'occupation_m_businessandfinancial','occupation_m_computerandmathematical','occupation_m_architectureandengineering',
        'occupation_m_lifephysicalsocialscience','occupation_m_communityandsocialservices','occupation_m_legal','occupation_m_educationtrainingandlibrary',
        'occupation_m_artsdesignentertainmentsportsandmedia','occupation_m_healthcarepractitionersandtechnical','occupation_m_healthcaresupport',
        'occupation_m_protectiveservice','occupation_m_foodprepandserving','occupation_m_buildingandgroundscleaningandmaintenance',
        'occupation_m_personalcareandservice','occupation_m_salesandrelated','occupation_m_officeandadmin','occupation_m_farmingfishingandforestry',
        'occupation_m_constructionandextraction','occupation_m_installationmaintenanceandrepair','occupation_m_production','occupation_m_transportation',
        'occupation_m_materialmoving','occupation_sexbyoccupationcivilianpop_totalfemale','occupation_f_management',
        'occupation_f_businessandfinancial','occupation_f_computerandmathematical','occupation_f_architectureandengineering','occupation_f_lifephysicalsocialscience',
        'occupation_f_communityandsocialservices','occupation_f_legal','occupation_f_educationtrainingandlibrary',
        'occupation_f_artsdesignentertainmentsportsandmedia','occupation_f_healthcarepractitionersandtechnical','occupation_f_healthcaresupport',
        'occupation_f_protectiveservice','occupation_f_foodprepandserving','occupation_f_buildingandgroundscleaningandmaintenance',
        'occupation_f_personalcareandservice','occupation_f_salesandrelated','occupation_f_officeandadmin','occupation_f_farmingfishingandforestry',
        'occupation_f_constructionandextraction','occupation_f_installationmaintenanceandrepair','occupation_f_production','occupation_f_transportation',
        'occupation_f_materialmoving']
data = data.drop(columns = cols)

### Employment Sector for Population 16+, and Percentages  
+ Private Sector  
+ Public Sector  
+ Self-Employed  
+ Non-Profit  
+ Unpaid Family Workers  

Make sure the team knows that private contains some self employed and nonprofit

In [15]:
data['Sector: Total Workers'] = data['classworker_total_series']
data['Sector:Private'] = data['classworker_privateforprofit']
data['Sector%:Private'] = percent(data['Sector:Private'], data['Sector: Total Workers'])
data['Sector:Public'] = data['classworker_localgovt']+data['classworker_stategovt']+data['classworker_federalgovt']
data['Sector%:Public'] = percent(data['Sector:Public'], data['Sector: Total Workers'])
data['Sector:Self-Employed'] = data['classworker_privateforprofit_selfemployedincorporatedbusiness']+data['classworker_selfemployednonincorporatedbusiness']
data['Sector%:Self-Employed'] = percent(data['Sector:Self-Employed'], data['Sector: Total Workers'])
data['Sector:Non-Profit'] = data['classworker_privatenotforprofit']
data['Sector%:Non-Profit'] = percent(data['Sector:Non-Profit'], data['Sector: Total Workers'])
data['Sector:Unpaid Family Workers'] = data['classworker_unpaidfamilyworker']
data['Sector%:Unpaid Family Workers'] = percent(data['Sector:Unpaid Family Workers'], data['Sector: Total Workers'])

cols = ['classworker_total_series','classworker_privateforprofit','classworker_privateforprofit_employee',
        'classworker_privateforprofit_selfemployedincorporatedbusiness','classworker_privatenotforprofit','classworker_localgovt','classworker_stategovt',
        'classworker_federalgovt','classworker_selfemployednonincorporatedbusiness','classworker_unpaidfamilyworker']
data = data.drop(columns = cols)

### Employment Status  
+ Employed  
+ Unemployed  
+ Unemployment Rate (unemployed divided by sum of employed and unemployed)

In [16]:
civemployed = [data['lfstatus_mciv_16to19_employed'],data['lfstatus_mciv_20to21_employed'],data['lfstatus_mciv_22to24_employed'],
               data['lfstatus_mciv_25to29_employed'],data['lfstatus_mciv_30to34_employed'],data['lfstatus_mciv_35to44_employed'],
               data['lfstatus_mciv_45to54_employed'],data['lfstatus_mciv_55to59_employed'],data['lfstatus_mciv_60to61_employed'],
               data['lfstatus_mciv_62to64_employed'],data['lfstatus_mciv_65to69_employed'],data['lfstatus_mciv_70to74_employed'],
               data['lfstatus_mciv_75+_employed'],data['lfstatus_fciv_16to19_employed'],data['lfstatus_fciv_20to21_employed'],
               data['lfstatus_fciv_22to24_employed'],data['lfstatus_fciv_25to29_employed'],data['lfstatus_fciv_30to34_employed'],
               data['lfstatus_fciv_35to44_employed'],data['lfstatus_fciv_45to54_employed'],data['lfstatus_fciv_55to59_employed'],
               data['lfstatus_fciv_60to61_employed'],data['lfstatus_fciv_62to64_employed'],data['lfstatus_fciv_65to69_employed'],
               data['lfstatus_fciv_70to74_employed'],data['lfstatus_fciv_75+_employed']]
data['Employed'] = sum(civemployed)
civunemployed = [data['lfstatus_mciv_16to19_unemployed'],data['lfstatus_mciv_20to21_unemployed'],data['lfstatus_mciv_22to24_unemployed'],
               data['lfstatus_mciv_25to29_unemployed'],data['lfstatus_mciv_30to34_unemployed'],data['lfstatus_mciv_35to44_unemployed'],
               data['lfstatus_mciv_45to54_unemployed'],data['lfstatus_mciv_55to59_unemployed'],data['lfstatus_mciv_60to61_unemployed'],
               data['lfstatus_mciv_62to64_unemployed'],data['lfstatus_mciv_65to69_unemployed'],data['lfstatus_mciv_70to74_unemployed'],
               data['lfstatus_mciv_75+_unemployed'],data['lfstatus_fciv_16to19_unemployed'],data['lfstatus_fciv_20to21_unemployed'],
               data['lfstatus_fciv_22to24_unemployed'],data['lfstatus_fciv_25to29_unemployed'],data['lfstatus_fciv_30to34_unemployed'],
               data['lfstatus_fciv_35to44_unemployed'],data['lfstatus_fciv_45to54_unemployed'],data['lfstatus_fciv_55to59_unemployed'],
               data['lfstatus_fciv_60to61_unemployed'],data['lfstatus_fciv_62to64_unemployed'],data['lfstatus_fciv_65to69_unemployed'],
               data['lfstatus_fciv_70to74_unemployed'],data['lfstatus_fciv_75+_unemployed']]
data['Unemployed'] = sum(civunemployed)
data['Unemployment Rate'] = data['Unemployed']/(data['Unemployed']+data['Employed'])

cols = ['lfstatus_total_sexbyagebyemploymentstatus16+_series','lfstatus_mciv_16to19','lfstatus_mciv_16to19_employed','lfstatus_mciv_16to19_unemployed',
        'lfstatus_mciv_20to21','lfstatus_mciv_20to21_employed','lfstatus_mciv_20to21_unemployed','lfstatus_mciv_22to24','lfstatus_mciv_22to24_employed',
        'lfstatus_mciv_22to24_unemployed','lfstatus_mciv_25to29','lfstatus_mciv_25to29_employed','lfstatus_mciv_25to29_unemployed','lfstatus_mciv_30to34',
        'lfstatus_mciv_30to34_employed','lfstatus_mciv_30to34_unemployed','lfstatus_mciv_35to44','lfstatus_mciv_35to44_employed','lfstatus_mciv_35to44_unemployed',
        'lfstatus_mciv_45to54','lfstatus_mciv_45to54_employed','lfstatus_mciv_45to54_unemployed','lfstatus_mciv_55to59','lfstatus_mciv_55to59_employed',
        'lfstatus_mciv_55to59_unemployed','lfstatus_mciv_60to61','lfstatus_mciv_60to61_employed','lfstatus_mciv_60to61_unemployed','lfstatus_mciv_62to64',
        'lfstatus_mciv_62to64_employed','lfstatus_mciv_62to64_unemployed','lfstatus_mciv_65to69','lfstatus_mciv_65to69_employed','lfstatus_mciv_65to69_unemployed',
        'lfstatus_mciv_70to74','lfstatus_mciv_70to74_employed','lfstatus_mciv_70to74_unemployed','lfstatus_mciv_75+','lfstatus_mciv_75+_employed',
        'lfstatus_mciv_75+_unemployed','lfstatus_fciv_16to19','lfstatus_fciv_16to19_employed','lfstatus_fciv_16to19_unemployed','lfstatus_fciv_20to21',
        'lfstatus_fciv_20to21_employed','lfstatus_fciv_20to21_unemployed','lfstatus_fciv_22to24','lfstatus_fciv_22to24_employed','lfstatus_fciv_22to24_unemployed',
        'lfstatus_fciv_25to29','lfstatus_fciv_25to29_employed','lfstatus_fciv_25to29_unemployed','lfstatus_fciv_30to34','lfstatus_fciv_30to34_employed',
        'lfstatus_fciv_30to34_unemployed','lfstatus_fciv_35to44','lfstatus_fciv_35to44_employed','lfstatus_fciv_35to44_unemployed','lfstatus_fciv_45to54',
        'lfstatus_fciv_45to54_employed','lfstatus_fciv_45to54_unemployed','lfstatus_fciv_55to59','lfstatus_fciv_55to59_employed','lfstatus_fciv_55to59_unemployed',
        'lfstatus_fciv_60to61','lfstatus_fciv_60to61_employed','lfstatus_fciv_60to61_unemployed','lfstatus_fciv_62to64','lfstatus_fciv_62to64_employed',
        'lfstatus_fciv_62to64_unemployed','lfstatus_fciv_65to69','lfstatus_fciv_65to69_employed','lfstatus_fciv_65to69_unemployed','lfstatus_fciv_70to74',
        'lfstatus_fciv_70to74_employed','lfstatus_fciv_70to74_unemployed','lfstatus_fciv_75+','lfstatus_fciv_75+_employed','lfstatus_fciv_75+_unemployed']
data = data.drop(columns = cols)

### Per Capita Income

In [17]:
data['Per Capita Income'] = data['percapita_income']
data = data.drop(columns = 'percapita_income')

## Costs Summary  

### Poverty, and Percentages  
+ Population for whom Poverty Status is Determined  
+ Population Living Below Poverty Level 

In [18]:
data['Poverty:Population for Whom Poverty Status is Determined'] = data['poverty_total_bysexbyage_series']
data['Poverty: Population Below Poverty Level'] = data['poverty_belowlevel']
data['Poverty%: Below Poverty Level'] = percent(data['Poverty: Population Below Poverty Level'], data['Poverty:Population for Whom Poverty Status is Determined'])

cols = ['poverty_total_bysexbyage_series','poverty_belowlevel']
data = data.drop(columns = cols)

### Housing Cost-Burden, and Percentages
+ Total Housing Units  
+ Cost Burdened Households 

I feel the need to check that there isn't overlap with owners w/w/o mortgage and renters... like is it all guaranteed that the one designation is owner-occupied?

+ Renter Occupied Units  
+ Cost Burdened Renters  
+ Owner Occupied Units  
+ Cost Burdened Homeowners  


+ Severe Housing Cost Burden (50% or More) 

In [19]:
data['CB:Total Housing Units'] = data['housingcost_total_selectedownercosts%hhincome_series']+data['housingcost_total_rent%hhincome_series']
allcostburden = [data['housingcost_%ownercost30to34.9_wmortgage'],data['housingcost_%ownercost35to39.9_wmortgage'],data['housingcost_%ownercost40to49.9_wmortgage'],
                 data['housingcost_%ownercost50+_wmortgage'],data['housingcost_%ownercost30to34.9_womortgage'],data['housingcost_%ownercost35to39.9_womortgage'],
                 data['housingcost_%ownercost40to49.9_womortgage'],data['housingcost_%ownercost50+_womortgage'],data['housingcost_%rentercost30to34.9'],
                 data['housingcost_%rentercost35to39.9'],data['housingcost_%rentercost40to49.9'],data['housingcost_%rentercost50+']]
data['CB:Cost Burdened Housholds'] = sum(allcostburden)
data['CB:Renter Occupied Units'] = data['housingcost_total_rent%hhincome_series']
rentercostburden = [data['housingcost_%rentercost30to34.9'],data['housingcost_%rentercost35to39.9'],
                    data['housingcost_%rentercost40to49.9'],data['housingcost_%rentercost50+']]
data['CB:Cost Burdened Renters'] = sum(rentercostburden)
data['CB%:Cost Burdened Renters'] = percent(data['CB:Cost Burdened Renters'], data['CB:Total Housing Units'])
data['CB:Owner Occupied Units'] = data['housingcost_total_selectedownercosts%hhincome_series']
ownercostburden = [data['housingcost_%ownercost30to34.9_wmortgage'],data['housingcost_%ownercost35to39.9_wmortgage'],data['housingcost_%ownercost40to49.9_wmortgage'],
                   data['housingcost_%ownercost50+_wmortgage'],data['housingcost_%ownercost30to34.9_womortgage'],data['housingcost_%ownercost35to39.9_womortgage'],
                   data['housingcost_%ownercost40to49.9_womortgage'],data['housingcost_%ownercost50+_womortgage']]
data['CB:Cost Burdened Homeowners'] = sum(ownercostburden)
data['CB%: Cost Burdened Homeowners'] = percent(data['CB:Cost Burdened Homeowners'], data['CB:Total Housing Units'])
severecostburden = [data['housingcost_%ownercost50+_womortgage'],data['housingcost_%ownercost50+_wmortgage'],data['housingcost_%rentercost50+']]
data['CB:Severe Cost Burdened Households'] = sum(severecostburden)
data['CB%:Severe Cost Burdened Households'] = percent(data['CB:Severe Cost Burdened Households'],data['CB:Total Housing Units'])

cols = ['housingcost_total_selectedownercosts%hhincome_series','housingcost_total%ownercostwmortgage_series','housingcost_%ownercost30to34.9_wmortgage',
        'housingcost_%ownercost35to39.9_wmortgage','housingcost_%ownercost40to49.9_wmortgage','housingcost_%ownercost50+_wmortgage',
        'housingcost_total%ownercostwomortgage_series','housingcost_%ownercost30to34.9_womortgage','housingcost_%ownercost35to39.9_womortgage',
        'housingcost_%ownercost40to49.9_womortgage','housingcost_%ownercost50+_womortgage','housingcost_total_rent%hhincome_series',
        'housingcost_%rentercost30to34.9','housingcost_%rentercost35to39.9','housingcost_%rentercost40to49.9','housingcost_%rentercost50+']
data = data.drop(columns = cols)

## Housing Summary

In [20]:
data['Total Households'] = data['units_allhousing']
data = data.drop(columns=['units_allhousing'])

### Median Home Value (Owner-Occupied Units)  
Remove for incorporated vs. unincorporated

In [21]:
data['Median Home Value Owner Occupied with Mortgage'] = data['housingcost_medvalue_ownerocc_wmortgage']
data['Median Home Value Owner Occupied without Mortgage'] = data['housingcost_medvalue_ownerocc_womortgage']

cols = ['housingcost_medvalue_ownerocc_wmortgage','housingcost_medvalue_ownerocc_womortgage']
data = data.drop(columns = cols)

### Median Gross Rent (Renter-Occupied Units)  
Remove for incorporated vs. unincorporated

In [22]:
data['Median Gross Rent'] = data['housingcost_mediangrossrent_renteroccupied']
data = data.drop(columns = 'housingcost_mediangrossrent_renteroccupied')

### Tenure, and Percentages  
+ Tenure Total Households  
+ Owners  
+ Renters

In [23]:
data['Tenure:Total Households'] = data['tenure_total_series']
data['Tenure:Owners'] = data['tenure_owneroccunits']
data['Tenure%:Owners'] = percent(data['Tenure:Owners'], data['Tenure:Total Households'])
data['Tenure:Renters'] = data['tenure_renteroccunits']
data['Tenure%:Renters'] = percent(data['Tenure:Renters'], data['Tenure:Total Households'])
cols = ['tenure_total_series','tenure_owneroccunits','tenure_renteroccunits']
data = data.drop(columns = cols)

### Occupancy Status, and Percentages  
+ Occupancy Total Households
+ Occupied  
+ Vacant  

In [24]:
data['Occupancy:Total Households'] = data['occupancy_total_series']
data['Occupancy:Occupied Units'] = data['occupancy_occupiedunits']
data['Occupancy%:Occupied Units'] = percent(data['Occupancy:Occupied Units'], data['Occupancy:Total Households'])
data['Occupancy:Vacant Units'] = data['occupancy_vacantunits']
data['Occupancy%:Vacant Units'] = percent(data['Occupancy:Vacant Units'], data['Occupancy:Total Households'])

cols = ['occupancy_total_series','occupancy_occupiedunits','occupancy_vacantunits']
data = data.drop(columns = cols)

### Units by Year Structure Built as of 2020, and Percentages
+ 2014 or Later  
+ 2010 to 2013  
+ 2000 to 2009  
+ 1990 to 1999  
+ 1980 to 1989  
+ 1970 to 1979  
+ 1960 to 1969  
+ 1950 to 1959  
+ 1939 or Earlier  

In [25]:
data['StructureAge:Total Structures'] = data['structures_total_yearbuilt_series']
data['StructureAge:Built 2014 or Later'] = data['structures_built2014orlater']
data['StructureAge%:Built 2014 or Later'] = percent(data['StructureAge:Built 2014 or Later'], data['StructureAge:Total Structures'])
data['StructureAge:Built 2010 to 2013'] = data['structures_built2010to2013']
data['StructureAge%:Built 2010 to 2013'] = percent(data['StructureAge:Built 2010 to 2013'], data['StructureAge:Total Structures'])
data['StructureAge:Built 2000 to 2009'] = data['structures_built2000to2009']
data['StructureAge%:Built 2000 to 2009'] = percent(data['StructureAge:Built 2000 to 2009'], data['StructureAge:Total Structures'])
data['StructureAge:Built 1990 to 1999'] = data['structures_built1990to1999']
data['StructureAge%:Built 1990 to 1999'] = percent(data['StructureAge:Built 1990 to 1999'], data['StructureAge:Total Structures'])
data['StructureAge:Built 1980 to 1989'] = data['structures_built1980to1989']
data['StructureAge%:Built 1980 to 1989'] = percent(data['StructureAge:Built 1980 to 1989'], data['StructureAge:Total Structures'])
data['StructureAge:Built 1970 to 1979'] = data['structures_built1970to1979']
data['StructureAge%: Built 1970 to 1979'] = percent(data['StructureAge:Built 1970 to 1979'], data['StructureAge:Total Structures'])
data['StructureAge:Built 1960 to 1969'] = data['structures_built1960to1969']
data['StructureAge%:Built 1960 to 1969'] = percent(data['StructureAge:Built 1960 to 1969'], data['StructureAge:Total Structures'])
data['StructureAge:Built 1950 to 1959'] = data['structures_built1950to1959']
data['StructureAge%:Built 1950 to 1959'] = percent(data['StructureAge:Built 1950 to 1959'], data['StructureAge:Total Structures'])
data['StructureAge:Built 1940 to 1949'] = data['structures_built1940to1949']
data['StructureAge%:Built 1940 to 1949'] = percent(data['StructureAge:Built 1940 to 1949'], data['StructureAge:Total Structures'])
data['StructureAge:Built 1939 or Earlier'] = data['structures_built1939orearlier']
data['StructureAge%:Built 1939 or Earlier'] = percent(data['StructureAge:Built 1939 or Earlier'], data['StructureAge:Total Structures'])

cols = ['structures_total_yearbuilt_series','structures_built2014orlater','structures_built2010to2013','structures_built2000to2009','structures_built1990to1999',
        'structures_built1980to1989','structures_built1970to1979','structures_built1960to1969','structures_built1950to1959','structures_built1940to1949',
        'structures_built1939orearlier']
data = data.drop(columns = cols)

### Median Year Residential Structure Built  
Remove for incorporated vs. unincorporated  
+ include the median age

In [26]:
data['StructureAge:Median Year Structure Built'] = data['structures_medianyearbuilt']  
data['StructureAge:Median Age of Structure'] = 2020 - data['StructureAge:Median Year Structure Built']
data = data.drop(columns = 'structures_medianyearbuilt')

### Housing Units and Units in Structure, and Percentages  
+ Housing Units: 1 Unit: 1, detached  
+ Housing Units: 1 Unit: 1, attached  
+ Housing Units: 2  
+ Housing Units: 3 or 4  
+ Housing Units: 5 to 9  
+ Housing Units: 10 to 19  
+ Housing Units: 20 to 49  
+ Housing Units: 50 or MOre  
+ Housing Units: Mobile Home  
+ Housing Units: Boat, RV, Van, etc.

In [27]:
data['Units:Total Series'] = data['units_total_series']
data['Units:1 Unit, Detached'] = data['units_one_detached']
data['Units%:1 Unit, Detached'] = percent(data['Units:1 Unit, Detached'], data['Units:Total Series'])
data['Units:1 Unit, Attached'] = data['units_one_attached']
data['Units%:1 Unit, Attached'] = percent(data['Units:1 Unit, Attached'], data['Units:Total Series'])
data['Units:2'] = data['units_two']
data['Units%:2'] = percent(data['Units:2'], data['Units:Total Series'])
data['Units:3 to 4'] = data['units_threetofour']
data['Units%:3 to 4'] = percent(data['Units:3 to 4'], data['Units:Total Series'])
data['Units:5 to 9'] = data['units_fivetonine']
data['Units%:5 to 9'] = percent(data['Units:5 to 9'], data['Units:Total Series'])
data['Units:10 to 19'] = data['units_tentonineteen']
data['Units%:10 to 19'] = percent(data['Units:10 to 19'], data['Units:Total Series'])
data['Units:20 to 49'] = data['units_twentytofortynine']
data['Units%:20 to 49'] = percent(data['Units:20 to 49'], data['Units:Total Series'])
data['Units:50 or More'] = data['units_fiftyormore']
data['Units%:50 or More'] = percent(data['Units:50 or More'], data['Units:Total Series'])
data['Units:Mobile Home'] = data['units_mobilehome']
data['Units%:Mobile Home'] = percent(data['Units:Mobile Home'], data['Units:Total Series'])
data['Units:Boat, RV, Van etc'] = data['units_boatrvvanetc']
data['Units%:Boat, RV, Van etc'] = percent(data['Units:Boat, RV, Van etc'], data['Units:Total Series'])

cols = ['units_total_series','units_one_detached','units_one_attached','units_two','units_threetofour','units_fivetonine','units_tentonineteen',
        'units_twentytofortynine','units_fiftyormore','units_mobilehome','units_boatrvvanetc']
data = data.drop(columns = cols)

## Transportation Summary

### Means of Transportation to Work for Workers 16+, and Percentages  
+ Transpo:Workers 16+  
+ Transpo:Car, Truck, or Van  
+ Transpo:Public Transportation  
+ Transpo:Motorcycle now Taxicab Motorcycle or Other  
+ Transpo:Bicycle  
+ Transpo:Walked  
+ Transpo:Other Means  
+ Transpo:Worked at Home  

In [28]:
data['Transpo:Workers Commuting'] = data['commute_total_meansoftransportationtowork_series']
data['Transpo:Car, Truck, or Van'] = data['commute_cartruckvan']
data['Transpo%:Car, Truck, or Van'] = percent(data['Transpo:Car, Truck, or Van'],data['Transpo:Workers Commuting'])
data['Transpo:Public Transportation'] = data['commute_publictransportation']
data['Transpo%:Public Transportation'] = percent(data['Transpo:Public Transportation'],data['Transpo:Workers Commuting'])
data['Transpo:Bicycle'] = data['commute_bicycle']
data['Transpo%:Bicycle'] = percent(data['Transpo:Bicycle'],data['Transpo:Workers Commuting'])
data['Transpo:Walk'] = data['commute_walk']
data['Transpo%:Walk'] = percent(data['Transpo:Walk'],data['Transpo:Workers Commuting'])
data['Transpo:Worked From Home'] = data['commute_workedfromhome']
data['Transpo%:Worked From Home'] = percent(data['Transpo:Worked From Home'],data['Transpo:Workers Commuting'])
data['Transpo:Taxi, Motorcycle, Other'] = data['commute_taxicabmotorcycleother']
data['Transpo%:Taxi, Motorcycle, Other'] = percent(data['Transpo:Taxi, Motorcycle, Other'],data['Transpo:Workers Commuting'])

cols = ['commute_total_meansoftransportationtowork_series','commute_cartruckvan','commute_cartruckvan_drovealone','commute_cartruckvan_carpooled',
        'commute_cartruckvan_carpooled_2ppl','commute_cartruckvan_carpooled_3ppl','commute_cartruckvan_carpooled_4ormoreppl','commute_publictransportation',
        'commute_publictransportation_bus','commute_publictransportation_subwayorelevatedrail','commute_publictransportation_longdistancetrainorcommuterrail',
        'commute_publictransportation_lightrailstreetcarortrolley','commute_publictransportation_ferryboat','commute_bicycle','commute_walk',
        'commute_taxicabmotorcycleother','commute_workedfromhome']
data = data.drop(columns = cols)

### Average Travel Time to Work  
Remove for incorporated and unincorporated areas

In [29]:
# data['Average Travel Time to Work'] = data['']
# data = data.drop(columns = '')

### Vehicle Ownership, and Percentages  
+ Vehicles:Households
+ Vehicles:None  
+ Vehicles:1  
+ Vehicles:2  
+ Vehicles:3  
+ Vehicles:4  
+ Vehicles:5+

In [30]:
data['Vehicles:Households'] = data['vehicles_tenurebyvehicles_total_series']
data['Vehicles:None'] = data['vehicles_ownerocc_novehicle']+data['vehicles_renterocc_novehicle']
data['Vehicles%:None'] = percent(data['Vehicles:None'],data['Vehicles:Households'])
data['Vehicles:One'] = data['vehicles_ownerocc_1vehicle']+data['vehicles_renterocc_1vehicle']
data['Vehicles%:One'] = percent(data['Vehicles:One'],data['Vehicles:Households'])
data['Vehicles:Two'] = data['vehicles_ownerocc_2vehicles']+data['vehicles_renterocc_2vehicles']
data['Vehicles%:Two'] = percent(data['Vehicles:Two'],data['Vehicles:Households'])
data['Vehicles:Three'] = data['vehicles_ownerocc_3vehicles']+data['vehicles_renterocc_3vehicles']
data['Vehicles%:Three'] = percent(data['Vehicles:Three'],data['Vehicles:Households'])
data['Vehicles:Four'] = data['vehicles_ownerocc_4vehicles']+data['vehicles_renterocc_4vehicles']
data['Vehicles%:Four'] = percent(data['Vehicles:Four'],data['Vehicles:Households'])
data['Vehicles:Five or More'] = data['vehicles_ownerocc_5+vehicles']+data['vehicles_renterocc_5+vehicles']
data['Vehicles%:Five or More'] = percent(data['Vehicles:Five or More'],data['Vehicles:Households'])

cols = ['vehicles_tenurebyvehicles_total_series','vehicles_none','vehicles_one','vehicles_two','vehicles_three','vehicles_four','vehicles_fiveormore',
        'vehicles_tenurebyvehicles_total_series','vehicles_ownerocc_total','vehicles_ownerocc_novehicle','vehicles_ownerocc_1vehicle',
        'vehicles_ownerocc_2vehicles','vehicles_ownerocc_3vehicles','vehicles_ownerocc_4vehicles','vehicles_ownerocc_5+vehicles','vehicles_renterocc_total',
        'vehicles_renterocc_novehicle','vehicles_renterocc_1vehicle','vehicles_renterocc_2vehicles','vehicles_renterocc_3vehicles','vehicles_renterocc_4vehicles',
        'vehicles_renterocc_5+vehicles', 'vehicles_total_series2']
data = data.drop(columns = cols)

### Travel Time by Mode  
+ Workers 16+ Not Working at Home  
+ Less than 10 Minutes  
+ 10 to 14 Minutes  
+ 15 to 19 Minutes  
+ 20 to 24 Minutes  
+ 25 to 29 Minutes  
+ 30 to 34 Minutes  
+ 35 to 44 Minutes  
+ 45 to 59 Minutes  
+ 60 or More Minutes  
+ Less than 10 Minutes: Car, Truck, or Van Drove Alone  
+ 10 to 14 Minutes: Car, Truck, or Van Drove Alone  
+ 15 to 19 Minutes: Car, Truck, or Van Drove Alone  
+ 20 to 24 Minutes: Car, Truck, or Van Drove Alone  
+ 25 to 29 Minutes: Car, Truck, or Van Drove Alone  
+ 30 to 34 Minutes: Car, Truck, or Van Drove Alone  
+ 35 to 44 Minutes: Car, Truck, or Van Drove Alone  
+ 45 to 59 Minutes: Car, Truck, or Van Drove Alone  
+ 60 or More Minutes: Car, Truck, or Van Drove Alone  
+ Less than 10 Minutes: Car, Truck, or Van Carpooled    
+ 10 to 14 Minutes: Car, Truck, or Van Carpooled    
+ 15 to 19 Minutes: Car, Truck, or Van Carpooled    
+ 20 to 24 Minutes: Car, Truck, or Van Carpooled    
+ 25 to 29 Minutes: Car, Truck, or Van Carpooled    
+ 30 to 34 Minutes: Car, Truck, or Van Carpooled    
+ 35 to 44 Minutes: Car, Truck, or Van Carpooled    
+ 45 to 59 Minutes: Car, Truck, or Van Carpooled    
+ 60 or More Minutes: Car, Truck, or Van Carpooled   
+ Less than 10 Minutes: Public Transportation (Excluding Taxicab)    
+ 10 to 14 Minutes: Public Transportation (Excluding Taxicab)    
+ 15 to 19 Minutes: Public Transportation (Excluding Taxicab)    
+ 20 to 24 Minutes: Public Transportation (Excluding Taxicab)    
+ 25 to 29 Minutes: Public Transportation (Excluding Taxicab)    
+ 30 to 34 Minutes: Public Transportation (Excluding Taxicab)    
+ 35 to 44 Minutes: Public Transportation (Excluding Taxicab)    
+ 45 to 59 Minutes: Public Transportation (Excluding Taxicab)    
+ 60 or More Minutes: Public Transportation (Excluding Taxicab)    
+ Less than 10 Minutes: Walked  
+ 10 to 14 Minutes: Walked  
+ 15 to 19 Minutes: Walked  
+ 20 to 24 Minutes: Walked  
+ 25 to 29 Minutes: Walked  
+ 30 to 34 Minutes: Walked  
+ 35 to 44 Minutes: Walked  
+ 45 to 59 Minutes: Walked  
+ 60 or More Minutes: Walked  
+ Less than 10 Minutes: Taxicab, Motorcycle, Bicycle, Other    
+ 10 to 14 Minutes: Taxicab, Motorcycle, Bicycle, Other    
+ 15 to 19 Minutes: Taxicab, Motorcycle, Bicycle, Other    
+ 20 to 24 Minutes: Taxicab, Motorcycle, Bicycle, Other    
+ 25 to 29 Minutes: Taxicab, Motorcycle, Bicycle, Other    
+ 30 to 34 Minutes: Taxicab, Motorcycle, Bicycle, Other    
+ 35 to 44 Minutes: Taxicab, Motorcycle, Bicycle, Other    
+ 45 to 59 Minutes: Taxicab, Motorcycle, Bicycle, Other    
+ 60 or More Minutes: Taxicab, Motorcycle, Bicycle, Other  

In [31]:
data['Commute:Workers 16+ Not Working at Home'] = data['traveltimemode_series_total']  
data['Commute:Less than 10 Minutes'] = data['traveltimemode_lessthan10']
data['Commute:10 to 14 Minutes'] = data['traveltimemode_10to14']
data['Commute:15 to 19 Minutes'] = data['traveltimemode_15to19']
data['Commute:20 to 24 Minutes'] = data['traveltimemode_20to24']
data['Commute:25 to 29 Minutes'] = data['traveltimemode_25to29']
data['Commute:30 to 34 Minutes'] = data['traveltimemode_30to34']
data['Commute:35 to 44 Minutes'] = data['traveltimemode_35to44']
data['Commute:45 to 59 Minutes'] = data['traveltimemode_45to59']
data['Commute:60 or More Minutes'] = data['traveltimemode_60ormore']
data['Commute:Car, Truck, Van Drove Alone:Less than 10 Minutes'] = data['traveltimemode_cartruckvan_drovealone_lessthan10']
data['Commute:Car, Truck, Van Drove Alone:10 to 14 Minutes'] = data['traveltimemode_cartruckvan_drovealone_10to14']
data['Commute:Car, Truck, Van Drove Alone:15 to 19 Minutes'] = data['traveltimemode_cartruckvan_drovealone_15to19']
data['Commute:Car, Truck, Van Drove Alone:20 to 24 Minutes'] = data['traveltimemode_cartruckvan_drovealone_20to24']
data['Commute:Car, Truck, Van Drove Alone:25 to 29 Minutes'] = data['traveltimemode_cartruckvan_drovealone_25to29']
data['Commute:Car, Truck, Van Drove Alone:30 to 34 Minutes'] = data['traveltimemode_cartruckvan_drovealone_30to34']
data['Commute:Car, Truck, Van Drove Alone:35 to 44 Minutes'] = data['traveltimemode_cartruckvan_drovealone_35to44']
data['Commute:Car, Truck, Van Drove Alone:45 to 59 Minutes'] = data['traveltimemode_cartruckvan_drovealone_45to59']
data['Commute:Car, Truck, Van Drove Alone:60 or More Minutes'] = data['traveltimemode_cartruckvan_drovealone_60ormore']
data['Commute:Car, Truck, Van Carpooled:Less than 10 Minutes'] = data['traveltimemode_cartruckvan_carpooled_lessthan10']
data['Commute:Car, Truck, Van Carpooled:10 to 14 Minutes'] = data['traveltimemode_cartruckvan_carpooled_10to14']
data['Commute:Car, Truck, Van Carpooled:15 to 19 Minutes'] = data['traveltimemode_cartruckvan_carpooled_15to19']
data['Commute:Car, Truck, Van Carpooled:20 to 24 Minutes'] = data['traveltimemode_cartruckvan_carpooled_20to24']
data['Commute:Car, Truck, Van Carpooled:25 to 29 Minutes'] = data['traveltimemode_cartruckvan_carpooled_25to29']
data['Commute:Car, Truck, Van Carpooled:30 to 34 Minutes'] = data['traveltimemode_cartruckvan_carpooled_30to34']
data['Commute:Car, Truck, Van Carpooled:35 to 44 Minutes'] = data['traveltimemode_cartruckvan_carpooled_35to44']
data['Commute:Car, Truck, Van Carpooled:45 to 59 Minutes'] = data['traveltimemode_cartruckvan_carpooled_45to59']
data['Commute:Car, Truck, Van Carpooled:60 or More Minutes'] = data['traveltimemode_cartruckvan_carpooled_60ormore']
data['Commute:Walked:Less than 10 Minutes'] = data['traveltimemode_walked_lessthan10']
data['Commute:Walked:10 to 14 Minutes'] = data['traveltimemode_walked_10to14']
data['Commute:Walked:15 to 19 Minutes'] = data['traveltimemode_walked_15to19']
data['Commute:Walked:20 to 24 Minutes'] = data['traveltimemode_walked_20to24']
data['Commute:Walked:25 to 29 Minutes'] = data['traveltimemode_walked_25to29']
data['Commute:Walked:30 to 34 Minutes'] = data['traveltimemode_walked_30to34']
data['Commute:Walked:35 to 44 Minutes'] = data['traveltimemode_walked_35to44']
data['Commute:Walked:45 to 59 Minutes'] = data['traveltimemode_walked_45to59']
data['Commute:Walked:60 or More Minutes'] = data['traveltimemode_walked_60ormore']
data['Commute:Walked:Less than 10 Minutes'] = data['traveltimemode_taximotorcyclebicycleorother_lessthan10']
data['Commute:Taxi, Motorcycle, Bicycle, Other:10 to 14 Minutes'] = data['traveltimemode_taximotorcyclebicycleorother_10to14']
data['Commute:Taxi, Motorcycle, Bicycle, Other:15 to 19 Minutes'] = data['traveltimemode_taximotorcyclebicycleorother_15to19']
data['Commute:Taxi, Motorcycle, Bicycle, Other:20 to 24 Minutes'] = data['traveltimemode_taximotorcyclebicycleorother_20to24']
data['Commute:Taxi, Motorcycle, Bicycle, Other:25 to 29 Minutes'] = data['traveltimemode_taximotorcyclebicycleorother_25to29']
data['Commute:Taxi, Motorcycle, Bicycle, Other:30 to 34 Minutes'] = data['traveltimemode_taximotorcyclebicycleorother_30to34']
data['Commute:Taxi, Motorcycle, Bicycle, Other:35 to 44 Minutes'] = data['traveltimemode_taximotorcyclebicycleorother_35to44']
data['Commute:Taxi, Motorcycle, Bicycle, Other:45 to 59 Minutes'] = data['traveltimemode_taximotorcyclebicycleorother_45to59']
data['Commute:Taxi, Motorcycle, Bicycle, Other:60 or More Minutes'] = data['traveltimemode_taximotorcyclebicycleorother_60ormore']

cols = ['traveltimemode_series_total','traveltimemode_lessthan10','traveltimemode_10to14','traveltimemode_15to19','traveltimemode_20to24','traveltimemode_25to29',
        'traveltimemode_30to34','traveltimemode_35to44','traveltimemode_45to59','traveltimemode_60ormore','traveltimemode_cartruckvan',
        'traveltimemode_cartruckvan_lessthan10','traveltimemode_cartruckvan_10to14','traveltimemode_cartruckvan_15to19','traveltimemode_cartruckvan_20to24',
        'traveltimemode_cartruckvan_25to29','traveltimemode_cartruckvan_30to34','traveltimemode_cartruckvan_35to44','traveltimemode_cartruckvan_45to59',
        'traveltimemode_cartruckvan_60ormore','traveltimemode_cartruckvan_drovealone','traveltimemode_cartruckvan_drovealone_lessthan10',
        'traveltimemode_cartruckvan_drovealone_10to14','traveltimemode_cartruckvan_drovealone_15to19','traveltimemode_cartruckvan_drovealone_20to24',
        'traveltimemode_cartruckvan_drovealone_25to29','traveltimemode_cartruckvan_drovealone_30to34','traveltimemode_cartruckvan_drovealone_35to44',
        'traveltimemode_cartruckvan_drovealone_45to59','traveltimemode_cartruckvan_drovealone_60ormore','traveltimemode_cartruckvan_carpooled',
        'traveltimemode_cartruckvan_carpooled_lessthan10','traveltimemode_cartruckvan_carpooled_10to14','traveltimemode_cartruckvan_carpooled_15to19',
        'traveltimemode_cartruckvan_carpooled_20to24','traveltimemode_cartruckvan_carpooled_25to29','traveltimemode_cartruckvan_carpooled_30to34',
        'traveltimemode_cartruckvan_carpooled_35to44','traveltimemode_cartruckvan_carpooled_45to59','traveltimemode_cartruckvan_carpooled_60ormore',
        'traveltimemode_publictransportationexcludingtaxi','traveltimemode_publictransportationexcludingtaxi_lessthan10',
        'traveltimemode_publictransportationexcludingtaxi_10to14','traveltimemode_publictransportationexcludingtaxi_15to19',
        'traveltimemode_publictransportationexcludingtaxi_20to24','traveltimemode_publictransportationexcludingtaxi_25to29',
        'traveltimemode_publictransportationexcludingtaxi_30to34','traveltimemode_publictransportationexcludingtaxi_35to44',
        'traveltimemode_publictransportationexcludingtaxi_45to59','traveltimemode_publictransportationexcludingtaxi_60ormore','traveltimemode_walked',
        'traveltimemode_walked_lessthan10','traveltimemode_walked_10to14','traveltimemode_walked_15to19','traveltimemode_walked_20to24',
        'traveltimemode_walked_25to29','traveltimemode_walked_30to34','traveltimemode_walked_35to44','traveltimemode_walked_45to59','traveltimemode_walked_60ormore',
        'traveltimemode_taximotorcyclebicycleorother','traveltimemode_taximotorcyclebicycleorother_lessthan10','traveltimemode_taximotorcyclebicycleorother_10to14',
        'traveltimemode_taximotorcyclebicycleorother_15to19','traveltimemode_taximotorcyclebicycleorother_20to24',
        'traveltimemode_taximotorcyclebicycleorother_25to29','traveltimemode_taximotorcyclebicycleorother_30to34',
        'traveltimemode_taximotorcyclebicycleorother_35to44','traveltimemode_taximotorcyclebicycleorother_45to59',
        'traveltimemode_taximotorcyclebicycleorother_60ormore']
data = data.drop(columns = cols)

## Miscellaneous Summary

### Veteran Status, and Percentages  
+ Civilian Population 18+  
+ Civilian Population 18+: Veteran  
+ Civilian Population 18+ Veteran: 18 to 64  
+ Civilian Population 18+ Veteran: 65 and Over  
+ Civilian Population 18+ Veteran: Nonveteran
+ Civilian Population 18+ Nonveteran: 18 to 64  
+ Civilian Population 18+ Nonveteran: 65 and Over 

In [32]:
data['Vet:Population 18+'] = data['veteran_total_series']
data['Vet:Veteran'] = data['veteran_total_veteran']
data['Vet%:Veteran'] = percent(data['Vet:Veteran'],data['Vet:Population 18+'])
data['Vet:Nonveteran'] = data['veteran_total_nonveteran']
data['Vet%:Nonveteran'] = percent(data['Vet:Nonveteran'],data['Vet:Population 18+'])
total1864 = [data['veteran_m18to34'],data['veteran_m35to54'],data['veteran_m55to64'],data['veteran_f18to34'],data['veteran_f35to54'],data['veteran_f55to64']]
vet1864 = [data['veteran_m18to34_veteran'],data['veteran_m35to54_veteran'],data['veteran_m55to64_veteran'],data['veteran_f18to34_veteran'],
           data['veteran_f35to54_veteran'],data['veteran_f55to64_veteran']]
data['Vet:Veteran 18 to 64'] = sum(vet1864)
data['Vet%:Veterans who are 18 to 64'] = percent(data['Vet:Veteran 18 to 64'], sum(total1864))
total65up = [data['veteran_m65to74'],data['veteran_m75+'],data['veteran_f65to74'],data['veteran_f75+']]
vet65up = [data['veteran_m65to74_veteran'],data['veteran_m75+_veteran'],data['veteran_f65to74_veteran'],data['veteran_f75+_veteran']]
data['Vet:Veteran 65 and Older'] = sum(vet65up)
data['Vet%:Veterans who are 65 and Older'] = percent(data['Vet:Veteran 65 and Older'], sum(total65up))
nonvet1864 = [data['veteran_m18to34_nonveteran'],data['veteran_m35to54_nonveteran'],data['veteran_m55to64_nonveteran'],data['veteran_f18to34_nonveteran'],
           data['veteran_f35to54_nonveteran'],data['veteran_f55to64_nonveteran']]
data['Vet:Nonveteran 18 to 64'] = sum(nonvet1864)
nonvet65up = [data['veteran_m65to74_nonveteran'],data['veteran_m75+_nonveteran'],data['veteran_f65to74_nonveteran'],data['veteran_f75+_nonveteran']]
data['Vet:Nonveteran 65 and Older'] = sum(vet65up)

cols = ['veteran_total_series','veteran_total_veteran','veteran_total_nonveteran','veteran_male','veteran_m_veteran','veteran_m_nonveteran','veteran_m18to34',
        'veteran_m18to34_veteran','veteran_m18to34_nonveteran','veteran_m35to54','veteran_m35to54_veteran','veteran_m35to54_nonveteran','veteran_m55to64',
        'veteran_m55to64_veteran','veteran_m55to64_nonveteran','veteran_m65to74','veteran_m65to74_veteran','veteran_m65to74_nonveteran','veteran_m75+',
        'veteran_m75+_veteran','veteran_m75+_nonveteran','veteran_female','veteran_f_veteran','veteran_f_nonveteran','veteran_f18to34','veteran_f18to34_veteran',
        'veteran_f18to34_nonveteran','veteran_f35to54','veteran_f35to54_veteran','veteran_f35to54_nonveteran','veteran_f55to64','veteran_f55to64_veteran',
        'veteran_f55to64_nonveteran','veteran_f65to74','veteran_f65to74_veteran','veteran_f65to74_nonveteran','veteran_f75+','veteran_f75+_veteran',
        'veteran_f75+_nonveteran']
data = data.drop(columns = cols)

### Disability Status  
+ Total Disabilities?  
+ Total Population  
+ Total Disabled Population and Percent of Population Disabled  
+ Hearing Difficulty  
+ Vision Difficulty  
+ Cognitive Difficulty  
+ Ambulatory Difficulty  
+ Self-Care Difficulty  
+ Independent Living Difficulty  
### Didn't pull Below 
+ Total, 1 Disability  
+ Total, 2 or More Disabilities  

In [33]:
data['Disability:Total Population'] = data['disability_sexbyage_total_series']
disability = [data['disability_mu5_disability'],data['disability_m5to17_disability'],data['disability_m18to34_disability'],data['disability_m35to64_disability'],
              data['disability_m65to74_disability'],data['disability_m75+_disability'],data['disability_fu5_disability'],data['disability_f5to17_disability'],
              data['disability_f18to34_disability'],data['disability_f35to64_disability'],data['disability_f65to74_disability'],data['disability_f75+_disability']]
data['Disability:With Disability'] = sum(disability)
data['Disability%:With Disability'] = percent(data['Disability:With Disability'],data['Disability:Total Population'])
hearing = [data['disability_mu5_hearingdifficulty'],data['disability_m5to17_hearingdifficulty'],data['disability_m18to34_hearingdifficulty'],
           data['disability_m35to64_hearingdifficulty'],data['disability_m65to74_hearingdifficulty'],data['disability_m75+_hearingdifficulty'],
           data['disability_fu5_hearingdifficulty'],data['disability_f5to17_hearingdifficulty'],data['disability_f18to34_hearingdifficulty'],
           data['disability_f35to64_hearingdifficulty'],data['disability_f65to74_hearingdifficulty'],data['disability_f75+_hearingdifficulty']]
data['Disability:Hearing Difficulty'] = sum(hearing)
data['Disability%:Hearing Difficulty'] = percent(data['Disability:Hearing Difficulty'], data['Disability:Total Population'])
vision = [data['disability_mu5_visiondifficulty'],data['disability_m5to17_visiondifficulty'],data['disability_m18to34_visiondifficulty'],
          data['disability_m35to64_visiondifficulty'],data['disability_m65to74_visiondifficulty'],data['disability_m75+_visiondifficulty'],
          data['disability_fu5_visiondifficulty'],data['disability_f5to17_visiondifficulty'],data['disability_f18to34_visiondifficulty'],
          data['disability_f35to64_visiondifficulty'],data['disability_f65to74_visiondifficulty'],data['disability_f75+_visiondifficulty']]
data['Disability:Vision Difficulty'] = sum(vision)
data['Disability%:Vision Difficulty'] = percent(data['Disability:Vision Difficulty'], data['Disability:Total Population'])
cognitive = [data['disability_m5to17_cognitivedifficulty'],data['disability_m18to34_cognitivedifficulty'],data['disability_m35to64_cognitivedifficulty'],
             data['disability_m65to74_cognitivedifficulty'],data['disability_m75+_cognitivedifficulty'],data['disability_f5to17_cognitivedifficulty'],
             data['disability_f18to34_cognitivedifficulty'],data['disability_f35to64_cognitivedifficulty'],data['disability_f65to74_cognitivedifficulty'],
             data['disability_f75+_cognitivedifficulty']]
data['Disability:Cognitive Difficulty'] = sum(cognitive)
data['Disability%:Cognitive Difficulty'] = percent(data['Disability:Cognitive Difficulty'], data['Disability:Total Population'])
ambulatory = [data['disability_m5to17_ambulatorydifficulty'],data['disability_m18to34_ambulatorydifficulty'],
             data['disability_m35to64_ambulatorydifficulty'],data['disability_m65to74_ambulatorydifficulty'],data['disability_m75+_ambulatorydifficulty'],
             data['disability_f5to17_ambulatorydifficulty'],data['disability_f18to34_ambulatorydifficulty'],
             data['disability_f35to64_ambulatorydifficulty'],data['disability_f65to74_ambulatorydifficulty'],data['disability_f75+_ambulatorydifficulty']]
data['Disability:Ambulatory Difficulty'] = sum(ambulatory)
data['Disability%:Ambulatory Difficulty'] = percent(data['Disability:Ambulatory Difficulty'], data['Disability:Total Population'])
selfcare = [data['disability_m5to17_selfcaredifficulty'],data['disability_m18to34_selfcaredifficulty'],data['disability_m35to64_selfcaredifficulty'],
            data['disability_m65to74_selfcaredifficulty'],data['disability_m75+_selfcaredifficulty'],data['disability_f5to17_selfcaredifficulty'],
            data['disability_f18to34_selfcaredifficulty'],data['disability_f35to64_selfcaredifficulty'],data['disability_f65to74_selfcaredifficulty'],
            data['disability_f75+_selfcaredifficulty']]
data['Disability:Self-Care Difficulty'] = sum(selfcare)
data['Disability%:Self-Care Difficulty'] = percent(data['Disability:Self-Care Difficulty'], data['Disability:Total Population'])
indliving = [data['disability_m18to34_independentlivingdifficulty'],
             data['disability_m35to64_independentlivingdifficulty'],data['disability_m65to74_independentlivingdifficulty'],
             data['disability_m75+_independentlivingdifficulty'],
             data['disability_f18to34_independentlivingdifficulty'],data['disability_f35to64_independentlivingdifficulty'],
             data['disability_f65to74_independentlivingdifficulty'],data['disability_f75+_independentlivingdifficulty']]
data['Disability:Independent Living Difficulty'] = sum(indliving)
data['Disability%:Independent Living Difficulty'] = percent(data['Disability:Independent Living Difficulty'], data['Disability:Total Population'])
data['Disability:Total Disabilities'] = sum(hearing)+sum(vision)+sum(cognitive)+sum(ambulatory)+sum(selfcare)+sum(indliving)

onedisability = [data['disability_u18_1disability'],data['disability_18to64_1disability'],data['disability_65+_1disability']]
data['Disability:Population with One Disability'] = sum(onedisability)
data['Disability%:Population with One Disability'] = percent(data['Disability:Population with One Disability'], data['disability_agebynumber_series_total'])
twomoredisability = [data['disability_u18_2ormoredisabilities'],data['disability_18to64_2ormoredisabilities'],data['disability_65+_2ormoredisabilities']]
data['Disability:Population with Two or More Disabilities'] = sum(twomoredisability)
data['Disability%:Population with Two or More Disabilities'] = percent(data['Disability:Population with Two or More Disabilities'], 
                                                                       data['disability_agebynumber_series_total'])

# cols = ['disability_agebynumber_series_total','disability_u18_total','disability_u18_1disability','disability_u18_2ormoredisabilities','disability_u18_nodisability',
#         'disability_18to64_total','disability_18to64_1disability','disability_18to64_2ormoredisabilities','disability_18to64_nodisability','disability_65+_total',
#         'disability_65+_1disability','disability_65+_2ormoredisabilities','disability_65+_nodisability',]

cols = ['disability_sexbyage_total_series','disability_totalmale','disability_mu5','disability_mu5_disability','disability_mu5_nodisability','disability_m5to17',
        'disability_m5to17_disability','disability_m5to17_nodisability','disability_m18to34','disability_m18to34_disability','disability_m18to34_nodisability',
        'disability_m35to64','disability_m35to64_disability','disability_m35to64_nodisability','disability_m65to74','disability_m65to74_disability',
        'disability_m65to74_nodisability','disability_m75+','disability_m75+_disability','disability_m75+_nodisability','disability_totalfemale','disability_fu5',
        'disability_fu5_disability','disability_fu5_nodisability','disability_f5to17','disability_f5to17_disability','disability_f5to17_nodisability',
        'disability_f18to34','disability_f18to34_disability','disability_f18to34_nodisability','disability_f35to64','disability_f35to64_disability',
        'disability_f35to64_nodisability','disability_f65to74','disability_f65to74_disability','disability_f65to74_nodisability','disability_f75+',
        'disability_f75+_disability','disability_f75+_nodisability','disability_sexbyage_hearing_total_series','disability_totalmale_hearing',
        'disability_mu5_hearing','disability_mu5_hearingdifficulty','disability_mu5_nohearingdifficulty','disability_m5to17_hearing',
        'disability_m5to17_hearingdifficulty','disability_m5to17_nohearingdifficulty','disability_m18to34_hearing','disability_m18to34_hearingdifficulty',
        'disability_m18to34_nohearingdifficulty','disability_m35to64_hearing','disability_m35to64_hearingdifficulty','disability_m35to64_nohearingdifficulty',
        'disability_m65to74_hearing','disability_m65to74_hearingdifficulty','disability_m65to74_nohearingdifficulty','disability_m75+_hearing',
        'disability_m75+_hearingdifficulty','disability_m75+_nohearingdifficulty','disability_totalfemale_hearing','disability_fu5_hearing',
        'disability_fu5_hearingdifficulty','disability_fu5_nohearingdifficulty','disability_f5to17_hearing','disability_f5to17_hearingdifficulty',
        'disability_f5to17_nohearingdifficulty','disability_f18to34_hearing','disability_f18to34_hearingdifficulty','disability_f18to34_nohearingdifficulty',
        'disability_f35to64_hearing','disability_f35to64_hearingdifficulty','disability_f35to64_nohearingdifficulty','disability_f65to74_hearing',
        'disability_f65to74_hearingdifficulty','disability_f65to74_nohearingdifficulty','disability_f75+_hearing','disability_f75+_hearingdifficulty',
        'disability_f75+_nohearingdifficulty','disability_sexbyage_vision_total_series','disability_totalmale_vision','disability_mu5_vision',
        'disability_mu5_visiondifficulty','disability_mu5_novisiondifficulty','disability_m5to17_vision','disability_m5to17_visiondifficulty',
        'disability_m5to17_novisiondifficulty','disability_m18to34_vision','disability_m18to34_visiondifficulty','disability_m18to34_novisiondifficulty',
        'disability_m35to64_vision','disability_m35to64_visiondifficulty','disability_m35to64_novisiondifficulty','disability_m65to74_vision',
        'disability_m65to74_visiondifficulty','disability_m65to74_novisiondifficulty','disability_m75+_vision','disability_m75+_visiondifficulty',
        'disability_m75+_novisiondifficulty','disability_totalfemale_vision','disability_fu5_vision','disability_fu5_visiondifficulty',
        'disability_fu5_novisiondifficulty','disability_f5to17_vision','disability_f5to17_visiondifficulty','disability_f5to17_novisiondifficulty',
        'disability_f18to34_vision','disability_f18to34_visiondifficulty','disability_f18to34_novisiondifficulty','disability_f35to64_vision',
        'disability_f35to64_visiondifficulty','disability_f35to64_novisiondifficulty','disability_f65to74_vision','disability_f65to74_visiondifficulty',
        'disability_f65to74_novisiondifficulty','disability_f75+_vision','disability_f75+_visiondifficulty','disability_f75+_novisiondifficulty',
        'disability_sexbyage_cognitive_total_series','disability_totalmale_cognitive','disability_m5to17_cognitive','disability_m5to17_cognitivedifficulty',
        'disability_m5to17_nocognitivedifficulty','disability_m18to34_cognitive','disability_m18to34_cognitivedifficulty','disability_m18to34_nocognitivedifficulty',
        'disability_m35to64_cognitive','disability_m35to64_cognitivedifficulty','disability_m35to64_nocognitivedifficulty','disability_m65to74_cognitive',
        'disability_m65to74_cognitivedifficulty','disability_m65to74_nocognitivedifficulty','disability_m75+_cognitive','disability_m75+_cognitivedifficulty',
        'disability_m75+_nocognitivedifficulty','disability_totalfemale_cognitive','disability_f5to17_cognitive','disability_f5to17_cognitivedifficulty',
        'disability_f5to17_nocognitivedifficulty','disability_f18to34_cognitive','disability_f18to34_cognitivedifficulty','disability_f18to34_nocognitivedifficulty',
        'disability_f35to64_cognitive','disability_f35to64_cognitivedifficulty','disability_f35to64_nocognitivedifficulty','disability_f65to74_cognitive',
        'disability_f65to74_cognitivedifficulty','disability_f65to74_nocognitivedifficulty','disability_f75+_cognitive','disability_f75+_cognitivedifficulty',
        'disability_f75+_nocognitivedifficulty','disability_sexbyage_ambulatory_total_series','disability_totalmale_ambulatory','disability_m5to17_ambulatory',
        'disability_m5to17_ambulatorydifficulty','disability_m5to17_noambulatorydifficulty','disability_m18to34_ambulatory','disability_m18to34_ambulatorydifficulty',
        'disability_m18to34_noambulatorydifficulty','disability_m35to64_ambulatory','disability_m35to64_ambulatorydifficulty',
        'disability_m35to64_noambulatorydifficulty','disability_m65to74_ambulatory','disability_m65to74_ambulatorydifficulty',
        'disability_m65to74_noambulatorydifficulty','disability_m75+_ambulatory','disability_m75+_ambulatorydifficulty','disability_m75+_noambulatorydifficulty',
        'disability_totalfemale_ambulatory','disability_f5to17_ambulatory','disability_f5to17_ambulatorydifficulty','disability_f5to17_noambulatorydifficulty',
        'disability_f18to34_ambulatory','disability_f18to34_ambulatorydifficulty','disability_f18to34_noambulatorydifficulty','disability_f35to64_ambulatory',
        'disability_f35to64_ambulatorydifficulty','disability_f35to64_noambulatorydifficulty','disability_f65to74_ambulatory',
        'disability_f65to74_ambulatorydifficulty','disability_f65to74_noambulatorydifficulty','disability_f75+_ambulatory','disability_f75+_ambulatorydifficulty',
        'disability_f75+_noambulatorydifficulty','disability_sexbyage_selfcare_total_series','disability_totalmale_selfcare','disability_m5to17_selfcare',
        'disability_m5to17_selfcaredifficulty','disability_m5to17_noselfcaredifficulty','disability_m18to34_selfcare','disability_m18to34_selfcaredifficulty',
        'disability_m18to34_noselfcaredifficulty','disability_m35to64_selfcare','disability_m35to64_selfcaredifficulty','disability_m35to64_noselfcaredifficulty',
        'disability_m65to74_selfcare','disability_m65to74_selfcaredifficulty','disability_m65to74_noselfcaredifficulty','disability_m75+_selfcare',
        'disability_m75+_selfcaredifficulty','disability_m75+_noselfcaredifficulty','disability_totalfemale_selfcare','disability_f5to17_selfcare',
        'disability_f5to17_selfcaredifficulty','disability_f5to17_noselfcaredifficulty','disability_f18to34_selfcare','disability_f18to34_selfcaredifficulty',
        'disability_f18to34_noselfcaredifficulty','disability_f35to64_selfcare','disability_f35to64_selfcaredifficulty','disability_f35to64_noselfcaredifficulty',
        'disability_f65to74_selfcare','disability_f65to74_selfcaredifficulty','disability_f65to74_noselfcaredifficulty','disability_f75+_selfcare',
        'disability_f75+_selfcaredifficulty','disability_f75+_noselfcaredifficulty','disability_sexbyage_independentliving_total_series',
        'disability_totalmale_independentliving','disability_m18to34_independentliving','disability_m18to34_independentlivingdifficulty',
        'disability_m18to34_noindependentlivingdifficulty','disability_m35to64_independentliving','disability_m35to64_independentlivingdifficulty',
        'disability_m35to64_noindependentlivingdifficulty','disability_m65to74_independentliving','disability_m65to74_independentlivingdifficulty',
        'disability_m65to74_noindependentlivingdifficulty','disability_m75+_independentliving','disability_m75+_independentlivingdifficulty',
        'disability_m75+_noindependentlivingdifficulty','disability_totalfemale_independentliving','disability_f18to34_independentliving',
        'disability_f18to34_independentlivingdifficulty','disability_f18to34_noindependentlivingdifficulty','disability_f35to64_independentliving',
        'disability_f35to64_independentlivingdifficulty','disability_f35to64_noindependentlivingdifficulty','disability_f65to74_independentliving',
        'disability_f65to74_independentlivingdifficulty','disability_f65to74_noindependentlivingdifficulty','disability_f75+_independentliving',
        'disability_f75+_independentlivingdifficulty','disability_f75+_noindependentlivingdifficulty','disability_agebynumber_series_total','disability_u18_total',
        'disability_u18_1disability','disability_u18_2ormoredisabilities','disability_u18_nodisability','disability_18to64_total','disability_18to64_1disability',
        'disability_18to64_2ormoredisabilities','disability_18to64_nodisability','disability_65+_total','disability_65+_1disability',
        'disability_65+_2ormoredisabilities','disability_65+_nodisability']
data = data.drop(columns = cols)

### Geographic Mobility, Moved, and Percentages  
+ Mobility: Total  
+ Mobility: Same House  
+ Mobility: Moved Same County  
+ Mobility: Moved Different County Same State  
+ Mobility: Moved Different State  
+ Mobility: Moved From Abroad

In [34]:
data['GeoMobility:Total'] = data['geo_total_series']
data['GeoMobility:Same House'] = data['geo_samehouse_total']
data['GeoMobility%:Same House'] = percent(data['GeoMobility:Same House'],data['GeoMobility:Total'])
data['GeoMobility:Moved Same County'] = data['geo_movedsamecounty_total']
data['GeoMobility%:Moved Same County'] = percent(data['GeoMobility:Moved Same County'],data['GeoMobility:Total'])
data['GeoMobility:Moved Different County Same State'] = data['geo_moveddifferentcountysamestate_total']
data['GeoMobility%:Moved Different County Same State'] = percent(data['GeoMobility:Moved Different County Same State'],data['GeoMobility:Total'])
data['GeoMobility:Moved Different State'] = data['geo_moveddifferentstate_total']
data['GeoMobility%:Moved Different State'] = percent(data['GeoMobility:Moved Different State'],data['GeoMobility:Total'])
data['GeoMobility:Moved From Abroad'] = data['geo_movedabroad_total']
data['GeoMobility%:Moved From Abroad'] = percent(data['GeoMobility:Moved From Abroad'],data['GeoMobility:Total'])

cols = ['geo_total_series','geo_1to4_total','geo_5to17_total','geo_18to19_total','geo_20to24_total','geo_25to29_total','geo_30to34_total','geo_35to39_total',
        'geo_40to44_total','geo_45to49_total','geo_50to54_total','geo_55to59_total','geo_60to64_total','geo_65to69_total','geo_70to74_total','geo_75+_total',
        'geo_samehouse_total','geo_55to59_samehouse','geo_60to64_samehouse','geo_65to69_samehouse','geo_70to74_samehouse','geo_75+_samehouse',
        'geo_movedsamecounty_total','geo_55to59_movedsamecounty','geo_60to64_movedsamecounty','geo_65to69_movedsamecounty','geo_70to74_movedsamecounty',
        'geo_75+_movedsamecounty','geo_moveddifferentcountysamestate_total','geo_55to59_moveddifferentcountysamestate','geo_60to64_moveddifferentcountysamestate',
        'geo_65to69_moveddifferentcountysamestate','geo_70to74_moveddifferentcountysamestate','geo_75+_moveddifferentcountysamestate','geo_moveddifferentstate_total',
        'geo_55to59_moveddifferentstate','geo_60to64_moveddifferentstate','geo_65to69_moveddifferentstate','geo_70to74_moveddifferentstate',
        'geo_75+_moveddifferentstate','geo_movedabroad_total','geo_55to59_movedabroad','geo_60to64_movedabroad','geo_65to69_movedabroad','geo_70to74_movedabroad',
        'geo_75+_movedabroad']
data = data.drop(columns = cols)

## Diversity Summary

### Foreign-Born Status  
+ Total Population  
+ Pop: Native Born  
+ Pop: Foreign Born and Percent  
+ Pop Foreign Born: Naturalized Citizen  
+ Pop Foreign Born: Not a Citizen

In [35]:
data['ForeignBorn:Total Population'] = data['fb_total_series']
data['ForeignBorn:Native Born'] = data['fb_nativeborn']
data['ForeignBorn:Foreign Born'] = data['fb_foreignborn']
data['ForeignBorn%:Foreign Born'] = percent(data['ForeignBorn:Foreign Born'], data['ForeignBorn:Total Population'])
data['ForeignBorn:Foreign Born Naturalized Citizen'] = data['fb_foreignborn_naturalizeduscitizen']
data['ForeignBorn%:Foreign Born Population that are Naturalized Citizen'] = percent(data['ForeignBorn:Foreign Born Naturalized Citizen'],
                                                                                    data['ForeignBorn:Foreign Born'])
data['ForeignBorn:Foreign Born Not a Citizen'] = data['fb_foreignborn_notauscitizen']
data['ForeignBorn%:Foreign Born Population that are Not Naturalized Citizen'] = percent(data['ForeignBorn:Foreign Born Not a Citizen'],
                                                                                        data['ForeignBorn:Foreign Born'])
cols = ['fb_total_series','fb_nativeborn','fb_foreignborn','fb_foreignborn_naturalizeduscitizen','fb_foreignborn_notauscitizen']
data = data.drop(columns = cols)

### Place of Birth for Foreign-Born Population  
+ Total Foreign Born Population  
+ Total from Europe  
+ Total from Asia  
+ Total from Africa  
+ Total from Oceania  
+ Total from Central + South America + Caribbean  
+ Total from Canada + Other Northern American  

In [36]:
data['Foreign Born Population'] = data['foreignborn_total']
data['ForeignBorn:Europe'] = data['fb_europe']
data['ForeignBorn:Asia'] = data['fb_asia']
data['ForeignBorn:Africa'] = data['fb_africa']
data['ForeignBorn:Oceania'] = data['fb_oceania']
data['ForeignBorn:Central & South America, Caribbean'] = data['fb_americas']
data['ForeignBorn:Canada & Other Northern American'] = data['fb_am_northern']

cols = ['foreignborn_total','fb_europe','fb_eur_northern','fb_eur_n_ireland','fb_eur_n_denmark','fb_eur_n_norway','fb_eur_n_sweden','fb_eur_n_uk',
        'fb_eur_n_excluding england and scotland','fb_eur_n_england','fb_eur_n_scotland','fb_eur_n_other','fb_eur_western','fb_eur_w_austria','fb_eur_w_belgium',
        'fb_eur_w_france','fb_eur_w_germany','fb_eur_w_netherlands','fb_eur_w_switzerland','fb_eur_w_other','fb_eur_southern','fb_eur_s_greece','fb_eur_s_italy',
        'fb_eur_s_portugal','fb_eur_s_azoresislands','fb_eur_s_spain','fb_eur_s_other','fb_eur_eastern','fb_eur_e_albania','fb_eur_e_belarus',
        'fb_eur_e_bosniaandherzegovina','fb_eur_e_bulgaria','fb_eur_e_croatia','fb_eur_e_czechoslovakia','fb_eur_e_hungary','fb_eur_e_latvia','fb_eur_e_lithuania',
        'fb_eur_e_macedonia','fb_eur_e_moldova','fb_eur_e_poland','fb_eur_e_romania','fb_eur_e_russia','fb_eur_e_serbia','fb_eur_e_ukraine','fb_eur_e_other',
        'fb_eur_nec','fb_asia','fb_as_eastern','fb_as_e_china','fb_as_e_chinanohongkongortaiwan','fb_as_e_hongkong','fb_as_e_taiwan','fb_as_e_japan','fb_as_e_korea',
        'fb_as_e_other','fb_as_southcentral','fb_as_sc_afghanistan','fb_as_sc_bangladesh','fb_as_sc_india','fb_as_sc_iran','fb_as_sc_kazakhstan','fb_as_sc_nepal',
        'fb_as_sc_pakistan','fb_as_sc_srilanka','fb_as_sc_uzbekistan','fb_as_sc_other','fb_as_southeastern','fb_as_se_burma','fb_as_se_cambodia','fb_as_se_indonesia',
        'fb_as_se_laos','fb_as_se_malaysia','fb_as_se_philippines','fb_as_se_singapore','fb_as_se_thailand','fb_as_se_vietnam','fb_as_se_other','fb_as_western',
        'fb_as_w_armenia','fb_as_w_iraq','fb_as_w_israel','fb_as_w_jordan','fb_as_w_kuwait','fb_as_w_lebanon','fb_as_w_saudiarabia','fb_as_w_syria','fb_as_w_turkey',
        'fb_as_w_yemen','fb_as_w_other','fb_as_nec','fb_africa','fb_af_eastern','fb_af_e_eritrea','fb_af_e_ethiopia','fb_af_e_kenya','fb_af_e_somalia',
        'fb_af_e_uganda','fb_af_e_zimbabwe','fb_af_e_other','fb_af_middle','fb_af_m_cameroon','fb_af_m_congo','fb_af_m_democraticrepublicofcongo','fb_af_m_other',
        'fb_af_northern','fb_af_n_egypt','fb_af_n_morocco','fb_af_n_sudan','fb_af_n_other','fb_af_southern','fb_af_s_southafrica','fb_af_s_other','fb_af_western',
        'fb_af_w_caboverde','fb_af_w_ghana','fb_af_w_liberia','fb_af_w_nigeria','fb_af_w_senegal','fb_af_w_sierraleone','fb_af_w_other','fb_af_nec','fb_oceania',
        'fb_oc_australianewzealandsubregion','fb_oc_australianewzealandsubregion_australia','fb_oc_australianewzealandsubregion_other','fb_oc_fiji','fb_oc_micronesia',
        'fb_oc_nec','fb_americas','fb_am_latinamerica','fb_am_la_caribbean','fb_am_la_cbb_bahamas','fb_am_la_cbb_barbados','fb_am_la_cbb_cuba','fb_am_la_cbb_dominica',
        'fb_am_la_cbb_dominicanrepublic','fb_am_la_cbb_grenada','fb_am_la_cbb_haiti','fb_am_la_cbb_jamaica','fb_am_la_cbb_stvincentandthegrenadines',
        'fb_am_la_cbb_trinidadandtobago','fb_am_la_cbb_westindies','fb_am_la_cbb_other','fb_am_la_centralamerica','fb_am_la_centr_belize','fb_am_la_centr_costarica',
        'fb_am_la_centr_elsalvador','fb_am_la_centr_guatemala','fb_am_la_centr_honduras','fb_am_la_centr_mexico','fb_am_la_centr_nicaragua','fb_am_la_centr_panama',
        'fb_am_la_centr_other','fb_am_la_southamerica','fb_am_la_s_argentina','fb_am_la_s_bolivia','fb_am_la_s_brazil','fb_am_la_s_chile','fb_am_la_s_columbia',
        'fb_am_la_s_ecuador','fb_am_la_s_guyana','fb_am_la_s_peru','fb_am_la_s_uruguay','fb_am_la_s_venezuela','fb_am_la_s_other','fb_am_northern','fb_am_n_canada',
        'fb_am_n_other']
data = data.drop(columns = cols)

### Language Spoken at Home by Ability to Speak English for Population 5+  
+ Language:Population 5 Years and Over  
+ Language:Speak Only English  
+ Language:Speak Spanish  
+ SpeakSpanish:Speak English Very Well  
+ SpeakSpanish:Speak English Well  
+ SpeakSpanish:Speak English Not Well  
+ SpeakSpanish:Speak English Not At All  
+ Language:Speak other Indo-European Languages
+ SpeakIndoEuro:Speak English Very Well  
+ SpeakIndoEuro:Speak English Well  
+ SpeakIndoEuro:Speak English Not Well  
+ SpeakIndoEuro:Speak English Not At All  
+ Speak Asian and Pacific Island Languages  
+ SpeakAsianPacificIsland:Speak English Very Well  
+ SpeakAsianPacificIsland:Speak English Well  
+ SpeakAsianPacificIsland:Speak English Not Well  
+ SpeakAsianPacificIsland:Speak English Not At All  
+ Language:Speak Other Languages
+ SpeakOtherLanguages:Speak English Very Well  
+ SpeakOtherLanguages:Speak English Well  
+ SpeakOtherLanguages:Speak English Not Well  
+ SpeakOtherLanguages:Speak English Not At All  

#### Percentages  
+ % Speak Only English  
+ % Speak Other Language  
+ % of Those that Speak Other Language and Speak English Not Well or At All

In [37]:
#all of the "very well's" say engish... fix on next go around

data['Language:Population 5 and Older'] = data['language_languagebyabilitytospeakenglish_total5+_series']
onlyenglish = [data['language_speakonlyenglish_5to17'],data['language_speakonlyenglish_18to64'],data['language_speakonlyenglish_65+']]
data['Language:Speak Only English'] = sum(onlyenglish)
data['Language%:Speak Only English'] = percent(data['Language:Speak Only English'],data['Language:Population 5 and Older'])
#spanish
speakspanish = [data['language_speakspanish_5to17'],data['language_speakspanish_18to64'],data['language_speakspanish_65+']]
data['Language:Speak Spanish'] = sum(speakspanish)
spanishverywell = [data['language_speakspanish_speakenglishverywell_5to17'],data['language_speakspanish_speakenglishverywell_18to64'],
                   data['language_speakspanish_speakenglishverywell_65+']]
data['Language:Spanish:Speak English Very Well'] = sum(spanishverywell)
spanishwell = [data['language_speakspanish_speakenglishwell_5to17'],data['language_speakspanish_speakenglishwell_18to64'],
                   data['language_speakspanish_speakenglishwell_65+']]
data['Language:Spanish:Speak English Well'] = sum(spanishwell)
spanishnotwell = [data['language_speakspanish_speakenglishnotwell_5to17'],data['language_speakspanish_speakenglishnotwell_18to64'],
                   data['language_speakspanish_speakenglishnotwell_65+']]
data['Language:Spanish:Speak English Not Well'] = sum(spanishnotwell)
spanishnotatall = [data['language_speakspanish_speakenglishnotatall_5to17'],data['language_speakspanish_speakenglishnotatall_18to64'],
                   data['language_speakspanish_speakenglishnotatall_65+']]
data['Language:Spanish:Speak English Not At All'] = sum(spanishnotatall)
#otherindoeuropeanlanguage
speakotherindoeuro = [data['language_speakotherindoeuro_5to17'],data['language_speakotherindoeuro_18to64'],data['language_speakotherindoeuro_65+']]
data['Language:Speak Other Indo-European Language'] = sum(speakotherindoeuro)
otherindoeuroverywell = [data['language_speakotherindoeuro_speakenglishverywell_5to17'],data['language_speakotherindoeuro_speakenglishverywell_18to64'],
                   data['language_speakotherindoeuro_speakenglishverywell_65+']]
data['Language:Other Indo-European:Speak English Very Well'] = sum(otherindoeuroverywell)
otherindoeurowell = [data['language_speakotherindoeuro_speakenglishwell_5to17'],data['language_speakotherindoeuro_speakenglishwell_18to64'],
                   data['language_speakotherindoeuro_speakenglishwell_65+']]
data['Language:Other Indo-European:Speak English Well'] = sum(otherindoeurowell)
otherindoeuronotwell = [data['language_speakotherindoeuro_speakenglishnotwell_5to17'],data['language_speakotherindoeuro_speakenglishnotwell_18to64'],
                   data['language_speakotherindoeuro_speakenglishnotwell_65+']]
data['Language:Other Indo-European:Speak English Not Well'] = sum(otherindoeuronotwell)
otherindoeuronotatall = [data['language_speakotherindoeuro_speakenglishnotatall_5to17'],data['language_speakotherindoeuro_speakenglishnotatall_18to64'],
                   data['language_speakotherindoeuro_speakenglishnotatall_65+']]
data['Language:Other Indo-European:Speak English Not At All'] = sum(otherindoeuronotatall)
#asianpacificislandlanguage
speakasianpacificisland = [data['language_speakasianpacificisland_5to17'],data['language_speakasianpacificisland_18to64'],data['language_speakasianpacificisland_65+']]
data['Language:Speak Asian or Pacific Island Language'] = sum(speakasianpacificisland)
asianpacificislandverywell = [data['language_speakasianpacificisland_speakenglishverywell_5to17'],data['language_speakasianpacificisland_speakenglishverywell_18to64'],
                   data['language_speakasianpacificisland_speakenglishverywell_65+']]
data['Language:Asian or Pacific Island:Speak English Very Well'] = sum(asianpacificislandverywell)
asianpacificislandwell = [data['language_speakasianpacificisland_speakenglishwell_5to17'],data['language_speakasianpacificisland_speakenglishwell_18to64'],
                   data['language_speakasianpacificisland_speakenglishwell_65+']]
data['Language:Asian or Pacific Island:Speak English Well'] = sum(asianpacificislandwell)
asianpacificislandnotwell = [data['language_speakasianpacificisland_speakenglishnotwell_5to17'],data['language_speakasianpacificisland_speakenglishnotwell_18to64'],
                   data['language_speakasianpacificisland_speakenglishnotwell_65+']]
data['Language:Asian or Pacific Island:Speak English Not Well'] = sum(asianpacificislandnotwell)
asianpacificislandnotatall = [data['language_speakasianpacificisland_speakenglishnotatall_5to17'],data['language_speakasianpacificisland_speakenglishnotatall_18to64'],
                   data['language_speakasianpacificisland_speakenglishnotatall_65+']]
data['Language:Asian or Pacific Island:Speak English Not At All'] = sum(asianpacificislandnotatall)
#other
speakother = [data['language_speakother_5to17'],data['language_speakother_18to64'],data['language_speakother_65+']]
data['Language:Speak Other Language'] = sum(speakother)
otherverywell = [data['language_speakother_speakenglishverywell_5to17'],data['language_speakother_speakenglishverywell_18to64'],
                   data['language_speakother_speakenglishverywell_65+']]
data['Language:Other:Speak English Very Well'] = sum(otherverywell)
otherwell = [data['language_speakother_speakenglishwell_5to17'],data['language_speakother_speakenglishwell_18to64'],
                   data['language_speakother_speakenglishwell_65+']]
data['Language:Other:Speak English Well'] = sum(otherwell)
othernotwell = [data['language_speakother_speakenglishnotwell_5to17'],data['language_speakother_speakenglishnotwell_18to64'],
                   data['language_speakother_speakenglishnotwell_65+']]
data['Language:Other:Speak English Not Well'] = sum(othernotwell)
othernotatall = [data['language_speakother_speakenglishnotatall_5to17'],data['language_speakother_speakenglishnotatall_18to64'],
                   data['language_speakother_speakenglishnotatall_65+']]
data['Language:Other:Speak English Not At All'] = sum(othernotatall)
languagenotenglish = [data['Language:Speak Spanish'],data['Language:Speak Other Indo-European Language'],data['Language:Speak Asian or Pacific Island Language'],
                     data['Language:Speak Other Language']]
data['Language%:Speak Language Besides English'] = percent(sum(languagenotenglish),data['Language:Population 5 and Older'])
englishnotwelloratall = [data['Language:Spanish:Speak English Not Well'],data['Language:Spanish:Speak English Not At All'],
                        data['Language:Other Indo-European:Speak English Not Well'],data['Language:Other Indo-European:Speak English Not At All'],
                        data['Language:Asian or Pacific Island:Speak English Not Well'],data['Language:Asian or Pacific Island:Speak English Not At All'],
                        data['Language:Other:Speak English Not Well'],data['Language:Other:Speak English Not At All']]
data['Language%:Of Those that Speak Another Language and Speak English Not Well or At All'] = percent(sum(englishnotwelloratall),sum(languagenotenglish))

cols = ['language_languagebyabilitytospeakenglish_total5+_series','language_total_5to17','language_speakonlyenglish_5to17','language_speakspanish_5to17',
        'language_speakspanish_speakenglishverywell_5to17','language_speakspanish_speakenglishwell_5to17','language_speakspanish_speakenglishnotwell_5to17',
        'language_speakspanish_speakenglishnotatall_5to17','language_speakotherindoeuro_5to17','language_speakotherindoeuro_speakenglishverywell_5to17',
        'language_speakotherindoeuro_speakenglishwell_5to17','language_speakotherindoeuro_speakenglishnotwell_5to17',
        'language_speakotherindoeuro_speakenglishnotatall_5to17','language_speakasianpacificisland_5to17',
        'language_speakasianpacificisland_speakenglishverywell_5to17','language_speakasianpacificisland_speakenglishwell_5to17',
        'language_speakasianpacificisland_speakenglishnotwell_5to17','language_speakasianpacificisland_speakenglishnotatall_5to17','language_speakother_5to17',
        'language_speakother_speakenglishverywell_5to17','language_speakother_speakenglishwell_5to17','language_speakother_speakenglishnotwell_5to17',
        'language_speakother_speakenglishnotatall_5to17','language_total_18to64','language_speakonlyenglish_18to64','language_speakspanish_18to64',
        'language_speakspanish_speakenglishverywell_18to64','language_speakspanish_speakenglishwell_18to64','language_speakspanish_speakenglishnotwell_18to64',
        'language_speakspanish_speakenglishnotatall_18to64','language_speakotherindoeuro_18to64','language_speakotherindoeuro_speakenglishverywell_18to64',
        'language_speakotherindoeuro_speakenglishwell_18to64','language_speakotherindoeuro_speakenglishnotwell_18to64',
        'language_speakotherindoeuro_speakenglishnotatall_18to64','language_speakasianpacificisland_18to64',
        'language_speakasianpacificisland_speakenglishverywell_18to64','language_speakasianpacificisland_speakenglishwell_18to64',
        'language_speakasianpacificisland_speakenglishnotwell_18to64','language_speakasianpacificisland_speakenglishnotatall_18to64',
        'language_speakother_18to64','language_speakother_speakenglishverywell_18to64','language_speakother_speakenglishwell_18to64',
        'language_speakother_speakenglishnotwell_18to64','language_speakother_speakenglishnotatall_18to64','language_total_65+','language_speakonlyenglish_65+',
        'language_speakspanish_65+','language_speakspanish_speakenglishverywell_65+','language_speakspanish_speakenglishwell_65+',
        'language_speakspanish_speakenglishnotwell_65+','language_speakspanish_speakenglishnotatall_65+','language_speakotherindoeuro_65+',
        'language_speakotherindoeuro_speakenglishverywell_65+','language_speakotherindoeuro_speakenglishwell_65+',
        'language_speakotherindoeuro_speakenglishnotwell_65+','language_speakotherindoeuro_speakenglishnotatall_65+','language_speakasianpacificisland_65+',
        'language_speakasianpacificisland_speakenglishverywell_65+','language_speakasianpacificisland_speakenglishwell_65+',
        'language_speakasianpacificisland_speakenglishnotwell_65+','language_speakasianpacificisland_speakenglishnotatall_65+','language_speakother_65+',
        'language_speakother_speakenglishverywell_65+','language_speakother_speakenglishwell_65+','language_speakother_speakenglishnotwell_65+',
        'language_speakother_speakenglishnotatall_65+']
data = data.drop(columns = cols)

In [38]:
data = data.drop(columns = ['StateFIPS', 'GeoFIPS'])

In [39]:
data.head()

,NAME,Population,HealthCoverage:Total Series,HealthCoverage:None,HealthCoverage%: None,HealthCoverage: With Healthcare Coverage,HealthCoverage%: With Healthcare Coverage,HealthCoverage: With Public Healthcare Coverage,HealthCoverage%: With Public Healthcare Coverage,HealthCoverage: With Private Healthcare Coverage,HealthCoverage%: With Private Healthcare Coverage,Male Under 5,Female Under 5,Male 5 to 9,Female 5 to 9,Male 10 to 14,Female 10 to 14,Male 15 to 17,Female 15 to 17,Male 18 to 24,Female 18 to 24,Male 25 to 34,Female 25 to 34,Male 35 to 44,Female 35 to 44,Male 45 to 54,Female 45 to 54,Male 55 to 64,Female 55 to 64,Male 65 to 74,Female 65 to 74,Male 75 to 84,Female 75 to 84,Male 85 and Older,Female 85 and Older,Age:Under 18,Age:18 to 24,Age:55 and Older,Age:65 and Older,White Alone,White Alone %,Black or African American Alone,Black or African American Alone %,American Indian Alaska Native Alone,American Indian Alaska Native Alone %,Asian Alone,Asian Alone %,Native Hawaiian Other Pacific Islander Alone,Native Hawaiian Other Pacific Islander Alone %,Some Other Race Alone,Some Other Race Alone %,Two or More Races,Two or More Races %,Hispanic or Latino,Hispanic or Latino %,Minority,Minority %,Total Households,Family Households,Family Households: Married Couple Family,Family Households: Not Married Couple Family,Family Households: Not Married Couple: Male no Spouse,Family Households: Not Married Couple: Female no Spouse,Nonfamily Households,Nonfamily Households: Householder Alone,Nonfamily Households: Householder not Alone,Average Household Size,Median Household Income,HHIncome:Total Households,"HHIncome:Less than 10,000","HHIncome%:Less than 10,000","HHIncome:10 to 14,999","HHIncome%:10 to 14,999","HHIncome:15 to 19,999","HHIncome%:15 to 19,999","HHIncome:20 to 24,999","HHIncome%:20 to 24,999","HHIncome:25 to 29,999","HHIncome%:25 to 29,999","HHIncome:30 to 34,999","HHIncome:%30 to 34,999","HHIncome:35 to 39,999","HHIncome%:35 to 39,999","HHIncome:40 to 44,999","HHIncome%:40 to 44,999","HHIncome:45 to 49,999","HHIncome%:45 to 49,999","HHIncome:50 to 59,999","HHIncome%:50 to 59,999","HHIncome:60 to 74,999","HHIncome%:60 to 74,999","HHIncome:75 to 99,999","HHIncome%:75 to 99,999","HHIncome:100 to 124,999","HHIncome%:100 to 124,999","HHIncome:125 to 149,999","HHIncome%:125 to 149,999","HHIncome:150 to 199,999","HHIncome%:150 to 199,999",HHIncome:200K or More,HHIncome%:200K or More,Ed:Population 25+ Educational Attainment,Ed:Less than High School,Ed%:Less than High School,Ed:High School Graduate or Equivalency,Ed%:High School Graduate or Equivalency,Ed%:High School Graduate or More,Ed:Some College,Ed%:Some College,Ed%:Some College or More,Ed:Associates,Ed%:Associates,Ed:Bachelors,Ed%:Bachelors,Ed%:Bachelors or More,Ed:Masters,Ed%:Masters or More,Ed%:Masters,Ed:Professional School Degree,Ed%:Professional School Degree,Ed%:Professional School Degree or More,Ed:Doctorate Degree,Ed%:Doctorate Degree,Ind:Employed Civilians 16+,"Ind:Agriculture, Forestry, Fishing and Hunting, and Mining","Ind%:Agriculture, Forestry, Fishing and Hunting, and Mining",Ind:Construction,Ind%:Construction,Ind:Manufacturing,Ind%:Manufacturing,Ind:Wholesale Trade,Ind%:Wholesale Trade,Ind:Retail Trade,Ind%:Retail Trade,"Ind:Transportation and Warehousing, Utilities","Ind%:Transportation and Warehousing, Utilities",Ind:Information,Ind%:Information,"Ind:Finance, Insurance, Real Estate and Rental and Leasing","Ind%:Finance, Insurance, Real Estate and Rental and Leasing","Ind:Professional, Scientific, Management, Administrative, and Waste Management Services","Ind%:Professional, Scientific, Management, Administrative, and Waste Management Services","Ind:Educational, Health and Social Services","Ind%:Educational, Health and Social Services","Ind:Arts, Entertainment, Recreation, Accommodation and Food Services","Ind%:Arts, Entertainment, Recreation, Accommodation and Food Services",Ind:Other Services (Except Public Administration),Ind%:Other Services (Except Publi